# Nonlinear Diffusion Interactive Model / Learning App — v2.3

**Research software release:** v2.3  
**Author:** Wook Kang  
**License:** MIT  
**Public-release metadata:** see `README.md` and `CITATION.cff`

Core forward model:

$$
\frac{\partial u}{\partial t}
=\nabla\cdot\left(\mathbf D(u)\nabla u\right),
\qquad
\mathbf D(u)=\operatorname{diag}\!\left(D_x(u),D_y(u),D_z(u)\right).
$$

The app is organized around

$$
\boxed{\text{Choose}\rightarrow\text{Simulate}\rightarrow\text{Observe}\rightarrow\text{Question}\rightarrow\text{Ask AI}}
$$

### v2.3 scope

- **Forward**: 1D / 2D / 3D.
- **FDM / FVM / FEM** remain available for the verified **isotropic** model.
- **Diagonal anisotropic forward** $D_x,D_y,D_z$ is implemented with **FVM** in 1D / 2D / 3D.
- **Inverse**: 1D / 2D FVM, using mean-response observations $\bar u(t)$.
- Inverse parameters are explicitly chosen as **Fixed** or **Estimate**.
- Estimated parameters can have initial guesses, bounds, and optional log-space optimization.
- Inverse diagnostics report fit residuals, bound proximity, Jacobian singular values / numerical rank, and a dedicated **Sensitivity / Identifiability panel** with time-resolved local sensitivities, parameter-strength comparison, sensitivity collinearity, and effective-rank diagnostics.

> Current anisotropy is **axis-aligned diagonal anisotropy**. A rotated tensor
> $\mathbf D=\mathbf Q\mathbf D_{\rm mat}\mathbf Q^T$ is a later extension.

### v2.3 record layer

- **Forward and inverse input summaries** are frozen at the moment a run starts.
- **Result summaries and time-series tables** are shown separately from plots.
- **Save Run** writes a reproducibility folder containing `input.json`, `input.csv`, `summary.csv`, `timeseries.csv`, and `report.md`.
- Optional compressed field output is saved as `field.npz`.
- **Load Run** restores saved input values into the widgets so the calculation can be reproduced.
- Optional **auto-save after run** is available for both Forward and Inverse modes.

### Reproducibility note

Run this notebook from the repository root so the local backend modules are importable. Saved run records capture inputs and outputs, but re-execution under a different Python/package environment can produce differences.


In [ ]:
# Dependencies and backend imports
import sys, io, math, time
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
from scipy.optimize import least_squares
from IPython.display import display, clear_output

try:
    import ipywidgets as widgets
except ImportError as e:
    raise ImportError("ipywidgets is required. Install it in the active Jupyter environment.") from e

HERE = Path.cwd()
if str(HERE) not in sys.path:
    sys.path.insert(0, str(HERE))

import fdm_backend as FDM
import fvm_backend as FVM
import fem_backend as FEM
import anisotropic_fvm_backend as AFVM

print("Backends loaded: scalar FDM/FVM/FEM + diagonal-anisotropic FVM")


In [ ]:

# ============================================================
# Common constitutive / IC / BC / adapter layer
# ============================================================

FACE_LABELS = {
    1: ["x-", "x+"],
    2: ["x-", "x+", "y-", "y+"],
    3: ["x-", "x+", "y-", "y+", "z-", "z+"],
}
AXIS_NAMES = ["x", "y", "z"]


def _safe_custom_eval(expr, local_vars):
    safe = {
        "np": np, "sin": np.sin, "cos": np.cos, "exp": np.exp,
        "sqrt": np.sqrt, "abs": np.abs, "pi": np.pi,
        "minimum": np.minimum, "maximum": np.maximum,
        **local_vars,
    }
    return eval(expr, {"__builtins__": {}}, safe)


def _coords_cell_center(shape, lengths):
    axes = tuple((np.arange(n)+0.5)*(L/n) for n,L in zip(shape,lengths))
    grids = np.meshgrid(*axes, indexing="ij")
    return axes, grids


def build_initial_condition(kind, coords, lengths, p):
    x = coords[0]
    y = coords[1] if len(coords)>=2 else np.zeros_like(x)
    z = coords[2] if len(coords)>=3 else np.zeros_like(x)
    Lx = lengths[0]
    Ly = lengths[1] if len(lengths)>=2 else 1.0
    Lz = lengths[2] if len(lengths)>=3 else 1.0
    if kind == "Constant":
        u = np.full_like(x, float(p["value"]), dtype=float)
    elif kind == "Linear x":
        u = float(p["left"]) + (float(p["right"])-float(p["left"]))*x/Lx
    elif kind == "Sine x":
        u = float(p["base"]) + float(p["amplitude"])*np.sin(np.pi*x/Lx)
    elif kind == "Gaussian":
        sigma=float(p["sigma"])
        r2=((x-0.5*Lx)/sigma)**2
        if len(coords)>=2: r2 += ((y-0.5*Ly)/sigma)**2
        if len(coords)>=3: r2 += ((z-0.5*Lz)/sigma)**2
        u=float(p["base"])+float(p["amplitude"])*np.exp(-0.5*r2)
    elif kind == "Piecewise x":
        u=np.where(x<float(p["split"]),float(p["left"]),float(p["right"]))
    elif kind == "Custom expression":
        u=_safe_custom_eval(str(p["expression"]),{"x":x,"y":y,"z":z,"Lx":Lx,"Ly":Ly,"Lz":Lz})
        u=np.broadcast_to(np.asarray(u,dtype=float),np.asarray(x).shape).copy()
    else:
        raise ValueError(f"Unknown IC type: {kind}")
    if not np.all(np.isfinite(u)):
        raise ValueError("Initial condition contains NaN or infinity.")
    return np.asarray(u,dtype=float)


def model_parameter_names(kind):
    return {
        "Constant": ["a"],
        "Linear": ["a","b"],
        "Exponential": ["a","b","u_ref"],
        "Power": ["a","b"],
        "Quadratic": ["a","b","c"],
        "Positive log-polynomial": ["c0","c1","c2"],
    }[kind]


def default_model_params(kind):
    return {
        "Constant": {"a":1e-10},
        "Linear": {"a":1e-10,"b":0.0},
        "Exponential": {"a":1e-10,"b":5.0,"u_ref":0.20},
        "Power": {"a":1e-10,"b":1.0},
        "Quadratic": {"a":1e-10,"b":0.0,"c":0.0},
        "Positive log-polynomial": {"c0":np.log(1e-10),"c1":5.0,"c2":0.0},
    }[kind].copy()


def build_diffusivity(kind, p):
    if kind == "Constant":
        a=float(p["a"])
        def raw(u): return np.zeros_like(np.asarray(u,dtype=float))+a
        def draw(u): return np.zeros_like(np.asarray(u,dtype=float))
        formula=f"D(u)={a:.6g}"
    elif kind == "Linear":
        a,b=float(p["a"]),float(p["b"])
        def raw(u): return a+b*np.asarray(u,dtype=float)
        def draw(u): return np.zeros_like(np.asarray(u,dtype=float))+b
        formula=f"D(u)={a:.6g}+({b:.6g})u"
    elif kind == "Exponential":
        a,b,ur=float(p["a"]),float(p["b"]),float(p["u_ref"])
        def raw(u): return a*np.exp(b*(np.asarray(u,dtype=float)-ur))
        def draw(u): return b*raw(u)
        formula=f"D(u)={a:.6g} exp[{b:.6g}(u-{ur:.6g})]"
    elif kind == "Power":
        a,b=float(p["a"]),float(p["b"])
        def raw(u):
            u=np.asarray(u,dtype=float)
            if np.any(u<=0): raise ValueError("Power D=a*u^b requires u>0.")
            return a*u**b
        def draw(u):
            u=np.asarray(u,dtype=float)
            if np.any(u<=0): raise ValueError("Power derivative requires u>0.")
            return a*b*u**(b-1)
        formula=f"D(u)={a:.6g}u^{b:.6g}"
    elif kind == "Quadratic":
        a,b,c=float(p["a"]),float(p["b"]),float(p["c"])
        def raw(u):
            u=np.asarray(u,dtype=float); return a+b*u+c*u*u
        def draw(u):
            u=np.asarray(u,dtype=float); return b+2*c*u
        formula=f"D(u)={a:.6g}+({b:.6g})u+({c:.6g})u²"
    elif kind == "Positive log-polynomial":
        c0,c1,c2=float(p["c0"]),float(p["c1"]),float(p["c2"])
        def raw(u):
            u=np.asarray(u,dtype=float); return np.exp(c0+c1*u+c2*u*u)
        def draw(u):
            u=np.asarray(u,dtype=float); return raw(u)*(c1+2*c2*u)
        formula=f"log D={c0:.6g}+({c1:.6g})u+({c2:.6g})u²"
    elif kind == "Custom expression":
        expr=str(p["expression"])
        def raw(u):
            u=np.asarray(u,dtype=float); return np.asarray(_safe_custom_eval(expr,{"u":u}),dtype=float)
        def draw(u):
            u=np.asarray(u,dtype=float); h=1e-6*np.maximum(1.0,np.abs(u))
            return (raw(u+h)-raw(u-h))/(2*h)
        formula=f"D(u)={expr}"
    else:
        raise ValueError(f"Unknown diffusivity model: {kind}")

    def D(u):
        v=np.asarray(raw(u),dtype=float)
        if not np.all(np.isfinite(v)) or np.any(v<=0):
            raise ValueError(f"Diffusivity positivity check failed; min D={np.nanmin(v):.6g}")
        return v
    def dD(u):
        v=np.asarray(draw(u),dtype=float)
        if not np.all(np.isfinite(v)): raise ValueError("dD/du contains NaN or infinity.")
        return v
    return D,dD,formula


def scalar_backend_bcs(method, dim, specs):
    if method in ("FDM","FVM"):
        B=FDM if method=="FDM" else FVM
        ov={}
        for face in FACE_LABELS[dim]:
            s=specs[face]
            if s["type"]=="Dirichlet": ov[face]=B.BC_D(s["value"])
            elif s["type"]=="Neumann": ov[face]=B.BC_N(s["flux"])
            else: ov[face]=B.BC_R(s["h"],s["env"])
        return B.make_box_bcs(dim,default=B.BC_N(0.0),overrides=ov)
    ov={}
    for face in FACE_LABELS[dim]:
        s=specs[face]
        if s["type"]=="Dirichlet": ov[face]=FEM.FEM_BC_D(s["value"])
        elif s["type"]=="Neumann": ov[face]=FEM.FEM_BC_N(s["flux"])
        else: ov[face]=FEM.FEM_BC_R(s["h"],s["env"])
    return FEM.make_fem_box_bcs(dim,default=FEM.FEM_BC_N(0.0),overrides=ov)


def anisotropic_bcs(dim,specs):
    out={}
    for face in FACE_LABELS[dim]:
        s=specs[face]
        if s["type"]=="Dirichlet": out[face]={"type":"dirichlet","value":float(s["value"])}
        elif s["type"]=="Neumann": out[face]={"type":"neumann","flux":float(s["flux"])}
        else: out[face]={"type":"robin","h":float(s["h"]),"env":float(s["env"])}
    return out


def _fem_volume_mean(result):
    measures=np.array([FEM.simplex_geometry(result["points"][e])[0] for e in result["elems"]])
    total=measures.sum(); means=[]
    for U in result["snapshots"]:
        em=np.array([np.mean(U[e]) for e in result["elems"]])
        means.append(np.sum(measures*em)/total)
    return np.asarray(means)


def solve_forward(cfg):
    method,dim=cfg["method"],cfg["dim"]
    shape=tuple(cfg["cells"][:dim]); lengths=tuple(cfg["lengths"][:dim])
    structure=cfg["D_structure"]

    if structure=="Diagonal anisotropic":
        if method!="FVM":
            raise ValueError("Diagonal anisotropic forward is currently implemented with FVM. Select FVM or switch to Isotropic.")
        axes,grids=_coords_cell_center(shape,lengths)
        U0=build_initial_condition(cfg["IC_kind"],grids,lengths,cfg["IC_params"])
        funcs=[]; formulas=[]
        for axis in range(dim):
            s=cfg["axis_D"][axis]
            f,_,formula=build_diffusivity(s["kind"],s["params"])
            f(U0); funcs.append(f); formulas.append(formula)
        res=AFVM.run_time(U0,shape,lengths,cfg["dt"],cfg["nsteps"],anisotropic_bcs(dim,cfg["bcs"]),funcs)
        res.update(method="FVM",D_structure=structure,D_axis_funcs=funcs,
                   D_formulas=[f"D_{AXIS_NAMES[i]}: {formulas[i]}" for i in range(dim)],
                   axes_common=axes,lengths_common=lengths)
        return res

    # verified scalar backends
    s=cfg["axis_D"][0]
    D,dD,formula=build_diffusivity(s["kind"],s["params"])
    bcs=scalar_backend_bcs(method,dim,cfg["bcs"])
    if method in ("FDM","FVM"):
        axes,grids=_coords_cell_center(shape,lengths)
        U0=build_initial_condition(cfg["IC_kind"],grids,lengths,cfg["IC_params"])
    else:
        points,_,_=FEM.make_simplex_box_nd(dim,lengths,shape)
        coords=tuple(points[:,i] for i in range(dim))
        U0=build_initial_condition(cfg["IC_kind"],coords,lengths,cfg["IC_params"])
    D(U0)
    if method=="FDM":
        res=FDM.solve_fdm_box_nd(shape,lengths,bcs,u0=U0,dt=cfg["dt"],nsteps=cfg["nsteps"],
                                  method=cfg["nonlinear_method"],D_func=D,dD_func=dD,D_kwargs=None)
        res["axes_common"]=res["axes"]
        res["mean_series"]=np.asarray([np.mean(s) for s in res["snapshots"]])
    elif method=="FVM":
        geom=FVM.fvm_geometry(shape,lengths)
        snaps,counts=FVM.run_time_fvm(U0,shape,lengths,cfg["dt"],cfg["nsteps"],bcs,
                                       method=cfg["nonlinear_method"],D_func=D,dD_func=dD,D_kwargs=None)
        res={"dim":dim,"geometry":geom,"bcs":bcs,"snapshots":snaps,"final":snaps[-1],
             "iteration_counts":counts,"axes_common":geom["axes"],
             "mean_series":np.asarray(snaps).reshape(len(snaps),-1).mean(axis=1)}
    else:
        res=FEM.solve_fem_box_nd(dim,lengths,shape,bcs,u0=U0,dt=cfg["dt"],nsteps=cfg["nsteps"],
                                  method=cfg["nonlinear_method"],D_func=D,dD_func=dD,D_kwargs=None)
        res["mean_series"]=_fem_volume_mean(res)
    res.update(method=method,D_structure="Isotropic",D_axis_funcs=[D]*dim,
               D_formulas=[f"D: {formula}"],lengths_common=lengths)
    res["times"]=np.arange(cfg["nsteps"]+1,dtype=float)*cfg["dt"]
    return res


def structured_flux(result):
    if result["method"] not in ("FDM","FVM"):
        return None,None
    U=np.asarray(result["final"])
    funcs=result["D_axis_funcs"]
    if len(funcs)==1 and U.ndim>1: funcs=funcs*U.ndim
    return AFVM.flux_components(U,result["lengths_common"],funcs)


In [ ]:

# ============================================================
# Reusable UI widget classes
# ============================================================
W=widgets

class DiffusivityWidget:
    """Forward constitutive editor for one spatial direction."""
    def __init__(self,label="D"):
        self.label=label
        self.kind=W.Dropdown(options=["Exponential","Constant","Linear","Power","Quadratic","Positive log-polynomial","Custom expression"],
                             value="Exponential",description=f"{label}(u)")
        self.a=W.FloatText(value=1e-10,description="a")
        self.b=W.FloatText(value=5.0,description="b")
        self.c=W.FloatText(value=0.0,description="c")
        self.ur=W.FloatText(value=0.20,description="u_ref")
        self.c0=W.FloatText(value=float(np.log(1e-10)),description="c0")
        self.c1=W.FloatText(value=5.0,description="c1")
        self.c2=W.FloatText(value=0.0,description="c2")
        self.expr=W.Text(value="1e-10*exp(5*(u-0.20))",description="expression",layout=W.Layout(width="640px"))
        self.param_box=W.VBox(); self.status=W.HTML()
        self.box=W.VBox([self.kind,self.param_box,self.status])
        self.kind.observe(self.refresh,names="value"); self.refresh()
    def refresh(self,*_):
        k=self.kind.value
        if k=="Constant": c=[self.a]; msg="a must be > 0."
        elif k=="Linear": c=[self.a,self.b]; msg="Runtime positivity check is applied."
        elif k=="Exponential": c=[self.a,self.b,self.ur]; msg="Positive when a > 0."
        elif k=="Power": c=[self.a,self.b]; msg="Requires u > 0."
        elif k=="Quadratic": c=[self.a,self.b,self.c]; msg="Runtime positivity check is applied."
        elif k=="Positive log-polynomial": c=[self.c0,self.c1,self.c2]; msg="Positive by construction."
        else: c=[self.expr]; msg="Custom expression uses runtime positivity validation."
        self.param_box.children=tuple(c); self.status.value=f"<span style='color:#666'>{msg}</span>"
    def spec(self):
        return {"kind":self.kind.value,"params":{"a":self.a.value,"b":self.b.value,"c":self.c.value,
                "u_ref":self.ur.value,"c0":self.c0.value,"c1":self.c1.value,"c2":self.c2.value,
                "expression":self.expr.value}}

class FaceBCWidget:
    def __init__(self,face,default_type="Robin"):
        self.face=face
        self.kind=W.Dropdown(options=["Dirichlet","Neumann","Robin"],value=default_type,description=face,layout=W.Layout(width="180px"))
        self.value=W.FloatText(value=0.10,description="value",layout=W.Layout(width="190px"))
        self.flux=W.FloatText(value=0.0,description="q_n",layout=W.Layout(width="190px"))
        self.h=W.FloatText(value=2e-8,description="h",layout=W.Layout(width="190px"))
        self.env=W.FloatText(value=0.10,description="u_env",layout=W.Layout(width="190px"))
        self.row=W.HBox(); self.kind.observe(self.refresh,names="value"); self.refresh()
    def refresh(self,*_):
        if self.kind.value=="Dirichlet": tail=[self.value]
        elif self.kind.value=="Neumann": tail=[self.flux]
        else: tail=[self.h,self.env]
        self.row.children=tuple([self.kind]+tail)
    def spec(self):
        return {"type":self.kind.value,"value":self.value.value,"flux":self.flux.value,
                "h":self.h.value,"env":self.env.value}

class InverseParameter:
    """Fixed/Estimate row. value means fixed value or optimization initial guess."""
    def __init__(self,name,value,lo,hi,estimate=False,log=False):
        self.name=name
        self.mode=W.Dropdown(options=["Fixed","Estimate"],value="Estimate" if estimate else "Fixed",description=name,layout=W.Layout(width="180px"))
        self.value=W.FloatText(value=value,description="value/init",layout=W.Layout(width="190px"))
        self.lo=W.FloatText(value=lo,description="lower",layout=W.Layout(width="190px"))
        self.hi=W.FloatText(value=hi,description="upper",layout=W.Layout(width="190px"))
        self.log=W.Checkbox(value=log,description="log-space",indent=False,layout=W.Layout(width="110px"))
        self.row=W.HBox([self.mode,self.value,self.lo,self.hi,self.log])
    def spec(self):
        return dict(name=self.name,estimate=self.mode.value=="Estimate",value=float(self.value.value),
                    lower=float(self.lo.value),upper=float(self.hi.value),log=bool(self.log.value))


def inverse_default_bounds(name,value):
    if name=="a": return (1e-14,1e-6,True)
    if name in ("b","c1","c2"): return (-100.0,100.0,False)
    if name=="u_ref": return (-2.0,2.0,False)
    if name in ("c0",): return (-40.0,-5.0,False)
    if name in ("c",): return (-1e-5,1e-5,False)
    return (-1e6,1e6,False)

class InverseDiffusivityWidget:
    def __init__(self,label="D_x",estimate_a=True):
        self.label=label
        self.kind=W.Dropdown(options=["Exponential","Constant","Linear","Power","Quadratic","Positive log-polynomial"],
                             value="Exponential",description=f"{label}(u)")
        self.rows={}; self.params_box=W.VBox(); self.box=W.VBox([self.kind,self.params_box])
        self.estimate_a=estimate_a
        self.kind.observe(self.refresh,names="value"); self.refresh()
    def refresh(self,*_):
        k=self.kind.value; defaults=default_model_params(k); old=self.rows
        self.rows={}
        for name in model_parameter_names(k):
            val=defaults[name]
            if name in old: val=old[name].value.value
            lo,hi,log=inverse_default_bounds(name,val)
            est=(name=="a" and self.estimate_a)
            if name in old:
                mode=old[name].mode.value; est=(mode=="Estimate"); log=old[name].log.value
                lo=old[name].lo.value; hi=old[name].hi.value
            self.rows[name]=InverseParameter(name,val,lo,hi,estimate=est,log=log)
        self.params_box.children=tuple(r.row for r in self.rows.values())
    def spec(self):
        return {"kind":self.kind.value,"params":{name:r.spec() for name,r in self.rows.items()}}

class InverseFaceBCWidget:
    def __init__(self,face):
        self.face=face
        self.kind=W.Dropdown(options=["Dirichlet","Neumann","Robin"],value="Robin",description=face,layout=W.Layout(width="180px"))
        self.dir_value=W.FloatText(value=0.10,description="u",layout=W.Layout(width="190px"))
        self.flux=W.FloatText(value=0.0,description="q_n",layout=W.Layout(width="190px"))
        self.env=W.FloatText(value=0.10,description="u_env",layout=W.Layout(width="190px"))
        self.h=InverseParameter("h",2e-8,1e-12,1e-4,estimate=False,log=True)
        self.box=W.VBox(); self.kind.observe(self.refresh,names="value"); self.refresh()
    def refresh(self,*_):
        if self.kind.value=="Dirichlet": ch=[W.HBox([self.kind,self.dir_value])]
        elif self.kind.value=="Neumann": ch=[W.HBox([self.kind,self.flux])]
        else: ch=[W.HBox([self.kind,self.env]),self.h.row]
        self.box.children=tuple(ch)
    def spec(self):
        return {"type":self.kind.value,"value":float(self.dir_value.value),"flux":float(self.flux.value),
                "env":float(self.env.value),"h":self.h.spec()}


In [ ]:

# ============================================================
# Forward UI
# ============================================================
METHOD=W.ToggleButtons(options=[("FVM (recommended)","FVM"),("FDM","FDM"),("FEM","FEM")],value="FVM",description="Method:")
DIM=W.ToggleButtons(options=[("1D",1),("2D",2),("3D",3)],value=1,description="Dimension:")

Lx=W.FloatText(value=0.05,description="Lx [m]"); Ly=W.FloatText(value=0.03,description="Ly [m]"); Lz=W.FloatText(value=0.02,description="Lz [m]")
Nx=W.IntText(value=60,description="Nx"); Ny=W.IntText(value=30,description="Ny"); Nz=W.IntText(value=12,description="Nz")
GEOM=W.VBox([W.HBox([Lx,Nx]),W.HBox([Ly,Ny]),W.HBox([Lz,Nz])])

IC_KIND=W.Dropdown(options=["Constant","Linear x","Sine x","Gaussian","Piecewise x","Custom expression"],value="Constant",description="IC")
IC_VALUE=W.FloatText(value=0.30,description="value"); IC_LEFT=W.FloatText(value=0.35,description="left"); IC_RIGHT=W.FloatText(value=0.20,description="right")
IC_BASE=W.FloatText(value=0.30,description="base"); IC_AMP=W.FloatText(value=0.05,description="amplitude"); IC_SIGMA=W.FloatText(value=0.008,description="sigma [m]")
IC_SPLIT=W.FloatText(value=0.025,description="split x [m]"); IC_EXPR=W.Text(value="0.30+0.05*sin(pi*x/Lx)",description="u0",layout=W.Layout(width="640px"))
IC_PARAMS=W.VBox()

D_STRUCTURE=W.ToggleButtons(options=["Isotropic","Diagonal anisotropic"],value="Isotropic",description="D tensor:")
FD=[DiffusivityWidget("D"),DiffusivityWidget("D_y"),DiffusivityWidget("D_z")]
FD[0].label="D_x"
D_AXIS_BOX=W.VBox()
D_NOTE=W.HTML()

FACES={f:FaceBCWidget(f) for f in ["x-","x+","y-","y+","z-","z+"]}
BC_BOX=W.VBox()
DT=W.FloatText(value=1800.0,description="dt [s]"); NSTEPS=W.IntText(value=8,description="steps")
NONLINEAR=W.Dropdown(options=[("Picard","picard"),("Newton","newton"),("Hybrid","hybrid")],value="picard",description="iteration")
RUN=W.Button(description="Run forward",button_style="success",icon="play")
FSTATUS=W.HTML(); FOUT=W.Output(); FQUEST=W.Textarea(layout=W.Layout(width="100%",height="145px"))


def update_forward_geometry(*_):
    d=DIM.value
    Ly.layout.display=Ny.layout.display="" if d>=2 else "none"
    Lz.layout.display=Nz.layout.display="" if d>=3 else "none"
    BC_BOX.children=tuple(FACES[f].row for f in FACE_LABELS[d])
    update_forward_D()


def update_forward_ic(*_):
    k=IC_KIND.value
    if k=="Constant": ch=[IC_VALUE]
    elif k=="Linear x": ch=[IC_LEFT,IC_RIGHT]
    elif k=="Sine x": ch=[IC_BASE,IC_AMP]
    elif k=="Gaussian": ch=[IC_BASE,IC_AMP,IC_SIGMA]
    elif k=="Piecewise x": ch=[IC_LEFT,IC_RIGHT,IC_SPLIT]
    else: ch=[IC_EXPR]
    IC_PARAMS.children=tuple(ch)


def update_forward_D(*_):
    d=DIM.value
    if D_STRUCTURE.value=="Isotropic":
        METHOD.disabled=False
        FD[0].kind.description="D(u)"
        D_AXIS_BOX.children=(FD[0].box,)
        D_NOTE.value="<span style='color:#666'>Scalar D(u): verified FDM / FVM / FEM backends are available.</span>"
    else:
        METHOD.value="FVM"; METHOD.disabled=True
        FD[0].kind.description="D_x(u)"
        children=[]
        for i in range(d): children.append(FD[i].box)
        D_AXIS_BOX.children=tuple(children)
        D_NOTE.value="<span style='color:#666'><b>Diagonal tensor:</b> FVM only in v2. Each active direction may use a different D_i(u).</span>"
    refresh_forward_questions()


def collect_forward():
    d=DIM.value; lengths=[Lx.value,Ly.value,Lz.value]; cells=[Nx.value,Ny.value,Nz.value]
    if any(v<=0 for v in lengths[:d]): raise ValueError("All active lengths must be >0.")
    if any(v<2 for v in cells[:d]): raise ValueError("Each active mesh count must be >=2.")
    if DT.value<=0 or NSTEPS.value<1: raise ValueError("dt>0 and steps>=1 required.")
    icp={"value":IC_VALUE.value,"left":IC_LEFT.value,"right":IC_RIGHT.value,"base":IC_BASE.value,"amplitude":IC_AMP.value,
         "sigma":IC_SIGMA.value,"split":IC_SPLIT.value,"expression":IC_EXPR.value}
    return {"method":METHOD.value,"dim":d,"lengths":lengths,"cells":cells,"IC_kind":IC_KIND.value,"IC_params":icp,
            "D_structure":D_STRUCTURE.value,"axis_D":[FD[i].spec() for i in range(d)],
            "bcs":{f:FACES[f].spec() for f in FACE_LABELS[d]},"dt":DT.value,"nsteps":NSTEPS.value,"nonlinear_method":NONLINEAR.value}


def plot_forward(res):
    U=np.asarray(res["final"]); d=res["dim"]; method=res["method"]
    plt.figure(figsize=(8,5))
    if method in ("FDM","FVM"):
        axes=res["axes_common"]
        if d==1:
            plt.plot(axes[0],U,marker="o",ms=3); plt.xlabel("x [m]"); plt.ylabel("u")
        elif d==2:
            plt.imshow(U.T,origin="lower",aspect="auto",extent=[0,res["lengths_common"][0],0,res["lengths_common"][1]])
            plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.colorbar(label="u")
        else:
            k=U.shape[2]//2
            plt.imshow(U[:,:,k].T,origin="lower",aspect="auto",extent=[0,res["lengths_common"][0],0,res["lengths_common"][1]])
            plt.xlabel("x [m]"); plt.ylabel("y [m]"); plt.colorbar(label="u"); plt.title(f"Final u, mid-z slice k={k}")
    else:
        pts=res["points"]
        if d==1:
            o=np.argsort(pts[:,0]); plt.plot(pts[o,0],U[o],marker="o",ms=3); plt.xlabel("x [m]"); plt.ylabel("u")
        elif d==2:
            plt.tricontourf(pts[:,0],pts[:,1],U,levels=20); plt.colorbar(label="u"); plt.xlabel("x [m]"); plt.ylabel("y [m]")
        else:
            ax=plt.axes(projection="3d"); sc=ax.scatter(pts[:,0],pts[:,1],pts[:,2],c=U,s=16); plt.colorbar(sc,label="u",shrink=.7)
    if not plt.gca().get_title(): plt.title(f"{method} final field")
    plt.tight_layout(); plt.show()

    plt.figure(figsize=(8,4)); plt.plot(res["times"],res["mean_series"],marker="o",ms=3)
    plt.xlabel("time [s]"); plt.ylabel("domain mean u"); plt.title("Domain-average response"); plt.tight_layout(); plt.show()

    grad,q=structured_flux(res)
    if q is not None:
        qmag=np.sqrt(sum(np.asarray(qi)**2 for qi in q))
        plt.figure(figsize=(8,4))
        if d==1:
            plt.plot(res["axes_common"][0],q[0]); plt.xlabel("x [m]"); plt.ylabel("q_x")
        else:
            arr=qmag if d==2 else qmag[:,:,qmag.shape[2]//2]
            plt.imshow(arr.T,origin="lower",aspect="auto"); plt.colorbar(label="|q|"); plt.xlabel("grid x"); plt.ylabel("grid y")
        plt.title("Flux diagnostic"); plt.tight_layout(); plt.show()

    print(f"Method={method}; dimension={d}D; tensor={res['D_structure']}")
    for s in res["D_formulas"]: print(s)
    print(f"Final min/max = {np.min(U):.6g} / {np.max(U):.6g}")
    print(f"Final mean = {res['mean_series'][-1]:.6g}")
    c=np.asarray(res["iteration_counts"])
    if c.size: print(f"Average nonlinear iterations/step = {np.mean(c):.3f}")


def refresh_forward_questions(*_):
    d=DIM.value; st=D_STRUCTURE.value; method=METHOD.value
    qs=[f"Why is {method} appropriate for this {d}D diffusion problem, and what changes with the alternative discretizations?",
        "How should I perform mesh and time-step convergence checks before interpreting the field physically?"]
    if st=="Diagonal anisotropic":
        qs += ["Can the chosen D_x(u), D_y(u), D_z(u) be justified as material-axis diffusivities, or is a rotated full tensor required?",
               "How sensitive are the field and mean response to each directional diffusivity?"]
    else:
        qs.append("Is the isotropic assumption physically defensible for this material and specimen geometry?")
    if any(FACES[f].kind.value=="Robin" for f in FACE_LABELS[d]):
        qs.append("How do surface transfer resistance and internal diffusivity compete in this geometry?")
    FQUEST.value="\n\n".join("• "+q for q in qs)


def run_forward_clicked(_):
    FSTATUS.value="<b>Running...</b>"
    with FOUT:
        clear_output(wait=True)
        try:
            t0=time.perf_counter(); res=solve_forward(collect_forward()); plot_forward(res)
            print(f"Wall time: {time.perf_counter()-t0:.3f} s")
            FSTATUS.value="<span style='color:green'><b>Completed.</b></span>"
        except Exception as e:
            print(type(e).__name__+":",e); FSTATUS.value=f"<span style='color:#b00020'><b>Error:</b> {e}</span>"

RUN.on_click(run_forward_clicked)
DIM.observe(update_forward_geometry,names="value"); IC_KIND.observe(update_forward_ic,names="value"); D_STRUCTURE.observe(update_forward_D,names="value")
METHOD.observe(refresh_forward_questions,names="value")
for f in FACES.values(): f.kind.observe(refresh_forward_questions,names="value")
update_forward_geometry(); update_forward_ic(); update_forward_D(); refresh_forward_questions()

FACC=W.Accordion(children=[GEOM,W.VBox([IC_KIND,IC_PARAMS]),W.VBox([D_STRUCTURE,D_AXIS_BOX,D_NOTE]),BC_BOX,W.HBox([DT,NSTEPS,NONLINEAR])])
for i,t in enumerate(["Geometry / mesh","Initial condition","Diffusivity tensor","Boundary conditions","Time / nonlinear solver"]): FACC.set_title(i,t)
FACC.selected_index=0
FORWARD_PANEL=W.VBox([W.HTML("<h3>Forward simulation</h3>"),METHOD,DIM,FACC,W.HBox([RUN,FSTATUS]),FOUT,
                      W.HTML("<h3>What should I ask AI?</h3>"),FQUEST])


In [ ]:
# ============================================================
# Inverse model layer: observation operator + parameter map + optimizer
# v2.2: synthetic round-trip verification + explicit local
#       sensitivity / identifiability diagnostics.
# ============================================================

def parse_observations(text,mode,time_unit,dry_mass=None):
    arr=np.genfromtxt(io.StringIO(text.strip()),delimiter=',' if ',' in text else None,comments='#')
    if arr.ndim==1: arr=arr[None,:]
    arr=arr[np.all(np.isfinite(arr),axis=1)]
    if arr.shape[0]<3 or arr.shape[1]<2: raise ValueError("Need at least 3 valid rows with two columns.")
    t=arr[:,0].astype(float); y=arr[:,1].astype(float)
    factor={"s":1.0,"min":60.0,"h":3600.0,"day":86400.0}[time_unit]; t=t*factor
    if mode=="time, mass":
        if dry_mass is None or dry_mass<=0: raise ValueError("Oven-dry mass must be >0.")
        y=(y-float(dry_mass))/float(dry_mass)
    o=np.argsort(t); t=t[o]; y=y[o]
    keep=np.r_[True,np.diff(t)>0]; t=t[keep]; y=y[keep]
    if np.any(t<0) or np.any(np.diff(t)<=0): raise ValueError("Times must be unique and non-negative.")
    return t,y


def parse_time_list(text,time_unit):
    clean=str(text).replace(',', ' ').replace(';',' ')
    vals=np.asarray([float(x) for x in clean.split()],dtype=float)
    if vals.size<3: raise ValueError("Synthetic verification needs at least 3 observation times.")
    vals=np.unique(vals)
    if vals[0] < 0: raise ValueError("Observation times must be non-negative.")
    factor={"s":1.0,"min":60.0,"h":3600.0,"day":86400.0}[time_unit]
    return vals, vals*factor


def _encoded_parameter(row_spec,name,target):
    if not row_spec["estimate"]: return None
    v,lo,hi=row_spec["value"],row_spec["lower"],row_spec["upper"]
    if not lo < hi: raise ValueError(f"{name}: lower bound must be < upper bound.")
    if not (lo <= v <= hi): raise ValueError(f"{name}: initial guess must lie within bounds.")
    if row_spec["log"]:
        if lo<=0 or v<=0 or hi<=0: raise ValueError(f"{name}: log-space requires positive initial guess and bounds.")
        return dict(name=name,target=target,log=True,z0=np.log(v),lo=np.log(lo),hi=np.log(hi))
    return dict(name=name,target=target,log=False,z0=v,lo=lo,hi=hi)


def compile_inverse_config(ui_cfg):
    dim=ui_cfg["dim"]
    axis_models=[]; unknowns=[]
    n_axis = 1 if ui_cfg["D_structure"]=="Isotropic" else dim
    for axis in range(n_axis):
        s=ui_cfg["axis_D"][axis]
        pd={name:row["value"] for name,row in s["params"].items()}
        axis_models.append({"kind":s["kind"],"params":pd})
        for pname,row in s["params"].items():
            u=_encoded_parameter(row,f"D_{AXIS_NAMES[axis]}.{pname}",( "D",axis,pname))
            if u: unknowns.append(u)

    bc_models={}
    for face,s in ui_cfg["bcs"].items():
        if s["type"]=="Robin":
            bc_models[face]={"type":"robin","h":s["h"]["value"],"env":s["env"]}
            u=_encoded_parameter(s["h"],f"{face}.h",("BC",face,"h"))
            if u: unknowns.append(u)
        elif s["type"]=="Dirichlet": bc_models[face]={"type":"dirichlet","value":s["value"]}
        else: bc_models[face]={"type":"neumann","flux":s["flux"]}

    if len(unknowns)==0: raise ValueError("Select at least one parameter as Estimate.")
    return axis_models,bc_models,unknowns


def decode_unknowns(z,axis_models,bc_models,unknowns):
    am=[{"kind":m["kind"],"params":dict(m["params"])} for m in axis_models]
    bm={k:dict(v) for k,v in bc_models.items()}
    physical=[]
    for zi,u in zip(z,unknowns):
        val=float(np.exp(zi) if u["log"] else zi); physical.append(val)
        if u["target"][0]=="D": am[u["target"][1]]["params"][u["target"][2]]=val
        else: bm[u["target"][1]][u["target"][2]]=val
    return am,bm,np.asarray(physical)


def inverse_predict(z,base,store_fields=False):
    am,bm,physical=decode_unknowns(z,base["axis_models"],base["bc_models"],base["unknowns"])
    funcs=[]; formulas=[]
    if base["D_structure"]=="Isotropic":
        f,_,form=build_diffusivity(am[0]["kind"],am[0]["params"]); funcs=[f]*base["dim"]; formulas=[form]*base["dim"]
    else:
        for m in am:
            f,_,form=build_diffusivity(m["kind"],m["params"]); funcs.append(f); formulas.append(form)
    U0=np.full(base["shape"],base["u0"],dtype=float)
    out=AFVM.solve_to_times(base["shape"],base["lengths"],base["times"],U0,bm,funcs,base["dt_max"],store_fields=store_fields)
    out["formulas"]=formulas; out["physical_unknowns"]=physical
    return out


def physical_predict_from_ui(cfg,times,store_fields=False):
    """Forward prediction using the current physical values, ignoring Fixed/Estimate flags."""
    dim=cfg["dim"]
    funcs=[]; formulas=[]
    n_axis=1 if cfg["D_structure"]=="Isotropic" else dim
    for axis in range(n_axis):
        s=cfg["axis_D"][axis]
        pd={name:row["value"] for name,row in s["params"].items()}
        f,_,form=build_diffusivity(s["kind"],pd)
        funcs.append(f); formulas.append(form)
    if cfg["D_structure"]=="Isotropic": funcs=[funcs[0]]*dim; formulas=[formulas[0]]*dim

    bm={}
    for face,s in cfg["bcs"].items():
        if s["type"]=="Robin": bm[face]={"type":"robin","h":s["h"]["value"],"env":s["env"]}
        elif s["type"]=="Dirichlet": bm[face]={"type":"dirichlet","value":s["value"]}
        else: bm[face]={"type":"neumann","flux":s["flux"]}
    shape=tuple(cfg["cells"][:dim]); lengths=tuple(cfg["lengths"][:dim])
    U0=np.full(shape,cfg["u0"],dtype=float)
    out=AFVM.solve_to_times(shape,lengths,np.asarray(times,dtype=float),U0,bm,funcs,cfg["dt_max"],store_fields=store_fields)
    out["formulas"]=formulas
    return out


def run_inverse(cfg):
    axis_models,bc_models,unknowns=compile_inverse_config(cfg)
    z0=np.array([u["z0"] for u in unknowns],dtype=float)
    lb=np.array([u["lo"] for u in unknowns],dtype=float); ub=np.array([u["hi"] for u in unknowns],dtype=float)
    base=dict(dim=cfg["dim"],shape=tuple(cfg["cells"][:cfg["dim"]]),lengths=tuple(cfg["lengths"][:cfg["dim"]]),
              D_structure=cfg["D_structure"],axis_models=axis_models,bc_models=bc_models,unknowns=unknowns,
              times=cfg["times"],obs=cfg["obs"],u0=cfg["u0"],dt_max=cfg["dt_max"])
    scale=np.maximum(np.abs(cfg["obs"]),cfg["residual_floor"]) if cfg["relative_residual"] else np.ones_like(cfg["obs"])
    eval_count={"n":0}
    def residual(z):
        eval_count["n"]+=1
        pred=inverse_predict(z,base,store_fields=False)["mean_series"]
        return (pred-cfg["obs"])/scale
    opt=least_squares(residual,z0,bounds=(lb,ub),method="trf",max_nfev=cfg["max_nfev"],xtol=cfg["xtol"],ftol=cfg["ftol"],gtol=cfg["gtol"])
    final=inverse_predict(opt.x,base,store_fields=True)
    raw_res=final["mean_series"]-cfg["obs"]
    svals=np.linalg.svd(np.asarray(opt.jac),compute_uv=False) if np.size(opt.jac) else np.array([])
    if svals.size:
        tol=svals[0]*max(opt.jac.shape)*np.finfo(float).eps
        rank=int(np.sum(svals>tol)); cond=float(svals[0]/svals[-1]) if svals[-1]>tol else np.inf
    else: rank=0; cond=np.inf
    r=dict(opt=opt,base=base,final=final,unknowns=unknowns,raw_residual=raw_res,
           singular_values=svals,rank=rank,condition=cond,evaluations=eval_count["n"],lb=lb,ub=ub,residual_scale=scale)
    r["diagnostics"]=assess_inverse_fit(r)
    return r


def assess_inverse_fit(r):
    obs=np.asarray(r["base"]["obs"],dtype=float)
    pred=np.asarray(r["final"]["mean_series"],dtype=float)
    res=pred-obs
    rmse=float(np.sqrt(np.mean(res**2)))
    mae=float(np.mean(np.abs(res)))
    dyn=float(np.ptp(obs))
    scale=max(dyn,0.1*max(float(np.max(np.abs(obs))),1e-12),1e-12)
    nrmse=rmse/scale
    sse=float(np.sum(res**2)); sst=float(np.sum((obs-np.mean(obs))**2))
    r2=float(1-sse/sst) if sst>1e-30 else np.nan

    z=np.asarray(r["opt"].x); lb=np.asarray(r["lb"]); ub=np.asarray(r["ub"])
    span=np.maximum(ub-lb,1e-300); pos=(z-lb)/span
    bound_flags=[]
    for u,p in zip(r["unknowns"],pos):
        if p<0.01: bound_flags.append((u["name"],"lower",float(p)))
        elif p>0.99: bound_flags.append((u["name"],"upper",float(p)))

    nz=res[np.abs(res)>max(1e-12,1e-6*scale)]
    same_sign=(np.mean(nz>0)>0.8 or np.mean(nz<0)>0.8) if nz.size>=3 else False
    if nrmse<=0.02 and not bound_flags:
        grade="GOOD"; color="green"; msg="fit is consistent with the observations at this scale"
    elif nrmse<=0.08 and not (bound_flags and same_sign):
        grade="REVIEW"; color="#b36b00"; msg="optimization converged, but fit/parameter diagnostics deserve review"
    else:
        grade="POOR"; color="#b00020"; msg="optimizer termination does not imply an adequate physical fit"
    return dict(rmse=rmse,mae=mae,nrmse=nrmse,r2=r2,bound_flags=bound_flags,
                systematic_same_sign=same_sign,grade=grade,color=color,message=msg)


# ============================================================
# Local sensitivity / identifiability diagnostics (v2.2)
# ============================================================
def compute_local_sensitivity_identifiability(r, rel_rank_tol=1e-6):
    """Compute local, parameter-scaled sensitivities from the optimizer Jacobian.

    The least-squares Jacobian is d[(prediction-observation)/residual_scale]/dz.
    We first recover d(prediction)/dz.  For log-space parameters z=ln(theta),
    this is already a fractional-parameter sensitivity d y / d ln(theta).
    For linear-space parameters, the derivative is multiplied by a characteristic
    parameter scale before normalization by the observed response scale.

    This is a LOCAL diagnostic around the fitted solution; it does not prove
    global uniqueness or structural identifiability.
    """
    rel_rank_tol=float(rel_rank_tol)
    if not (0 < rel_rank_tol < 1):
        raise ValueError("relative rank tolerance must lie between 0 and 1.")

    Jres=np.asarray(r["opt"].jac,dtype=float)
    if Jres.ndim==1:
        Jres=Jres[:,None]
    m,n=Jres.shape
    if n != len(r["unknowns"]):
        raise ValueError("Optimizer Jacobian column count does not match the estimated parameter set.")

    residual_scale=np.asarray(r.get("residual_scale",np.ones(m)),dtype=float).reshape(-1)
    if residual_scale.size != m:
        raise ValueError("Residual scale length does not match observation count.")
    Jpred_z=Jres*residual_scale[:,None]

    obs=np.asarray(r["base"]["obs"],dtype=float)
    pred=np.asarray(r["final"]["mean_series"],dtype=float)
    yscale=max(float(np.ptp(obs)),0.1*max(float(np.max(np.abs(obs))),1e-12),1e-12)
    physical=np.asarray(r["final"]["physical_unknowns"],dtype=float)

    S=np.zeros_like(Jpred_z,dtype=float)
    parameter_scales=[]
    sensitivity_coordinates=[]
    for j,(u,theta) in enumerate(zip(r["unknowns"],physical)):
        if u["log"]:
            # z=ln(theta): Jpred_z = theta * dy/dtheta
            pscale=1.0
            S[:,j]=Jpred_z[:,j]/yscale
            coord="d y / d ln(theta)"
        else:
            lo=float(r["lb"][j]); hi=float(r["ub"][j])
            pscale=max(abs(float(theta)),0.1*abs(hi-lo),1e-15)
            S[:,j]=(Jpred_z[:,j]*pscale)/yscale
            coord=f"scale={pscale:.3g}"
        parameter_scales.append(pscale)
        sensitivity_coordinates.append(coord)

    rms=np.sqrt(np.mean(S*S,axis=0)) if m else np.zeros(n)
    max_rms=float(np.max(rms)) if rms.size else 0.0
    rel_strength=rms/max(max_rms,1e-300)

    col_norm=np.linalg.norm(S,axis=0)
    C=np.eye(n,dtype=float)
    for i in range(n):
        for j in range(n):
            den=col_norm[i]*col_norm[j]
            C[i,j]=float(np.dot(S[:,i],S[:,j])/den) if den>1e-300 else np.nan

    sv=np.linalg.svd(S,compute_uv=False) if S.size else np.array([])
    if sv.size and sv[0]>0:
        svrel=sv/sv[0]
        eff_rank=int(np.sum(svrel>=rel_rank_tol))
        condition=float(sv[0]/sv[-1]) if sv[-1]>sv[0]*rel_rank_tol else np.inf
    else:
        svrel=np.array([]); eff_rank=0; condition=np.inf

    pairs=[]
    if n>=2:
        for i in range(n):
            for j in range(i+1,n):
                cij=C[i,j]
                if np.isfinite(cij): pairs.append((abs(float(cij)),float(cij),i,j))
        pairs.sort(reverse=True)
    max_abs_collinearity=pairs[0][0] if pairs else 0.0

    low_sensitivity=[r["unknowns"][i]["name"] for i,x in enumerate(rel_strength) if x<0.01]
    high_collinearity=[]
    for a,cij,i,j in pairs:
        if a>=0.95:
            high_collinearity.append((r["unknowns"][i]["name"],r["unknowns"][j]["name"],cij))

    # Heuristic LOCAL grade; thresholds are diagnostic guides, not proofs.
    if n==1:
        if rms[0] <= 1e-10:
            grade="POOR"; reason="the estimated parameter has essentially zero local sensitivity"
        else:
            grade="GOOD"; reason="the single estimated parameter has non-zero local sensitivity"
    elif eff_rank < n or max_abs_collinearity>=0.995 or not np.isfinite(condition) or condition>1e6:
        grade="POOR"; reason="local sensitivity directions are rank-deficient or nearly indistinguishable"
    elif max_abs_collinearity>=0.95 or condition>1e3 or (rel_strength.size and np.min(rel_strength)<0.01):
        grade="REVIEW"; reason="local sensitivities show strong imbalance or parameter collinearity"
    else:
        grade="GOOD"; reason="local sensitivity directions are reasonably distinct at this solution"

    return dict(
        S=S, J_prediction_z=Jpred_z, yscale=yscale, rms=rms, relative_strength=rel_strength,
        collinearity=C, singular_values=sv, relative_singular_values=svrel,
        effective_rank=eff_rank, condition=condition, rel_rank_tol=rel_rank_tol,
        low_sensitivity=low_sensitivity, high_collinearity=high_collinearity,
        max_abs_collinearity=max_abs_collinearity, parameter_scales=np.asarray(parameter_scales),
        sensitivity_coordinates=sensitivity_coordinates, grade=grade, reason=reason,
        times=np.asarray(r["base"]["times"],dtype=float), prediction=pred, observation=obs,
    )


In [ ]:
# ============================================================
# Inverse UI — 1D / 2D mean-response fitting
# v2.2 default workflow: synthetic round-trip verification first,
# then experimental fitting + local sensitivity/identifiability review.
# ============================================================
IDIM=W.ToggleButtons(options=[("1D",1),("2D",2)],value=1,description="Dimension:")
ILx=W.FloatText(value=0.02,description="Lx [m]"); ILy=W.FloatText(value=0.02,description="Ly [m]")
INx=W.IntText(value=30,description="Nx"); INy=W.IntText(value=20,description="Ny")
IU0=W.FloatText(value=0.30,description="initial u")
IDT=W.FloatText(value=1800.0,description="dt max [s]")

IDSTRUCT=W.ToggleButtons(options=["Isotropic","Diagonal anisotropic"],value="Isotropic",description="D tensor:")
IDW=[InverseDiffusivityWidget("D_x",True),InverseDiffusivityWidget("D_y",False)]
# Start inverse verification with the simplest identifiable model.
IDW[0].kind.value="Constant"
IDBOX=W.VBox()
IBCS={f:InverseFaceBCWidget(f) for f in ["x-","x+","y-","y+"]}; IBCBOX=W.VBox()

OBS_SOURCE=W.ToggleButtons(options=[("Synthetic round-trip","Synthetic"),("Experimental data","Experimental")],
                           value="Synthetic",description="Observation:")
DATA_MODE=W.Dropdown(options=["time, mean u","time, mass"],value="time, mean u",description="Data")
TIME_UNIT=W.Dropdown(options=["s","min","h","day"],value="h",description="time unit")
DRY_MASS=W.FloatText(value=1.0,description="dry mass")
DATA_TEXT=W.Textarea(value="",description="data",layout=W.Layout(width="650px",height="190px"))
UPLOAD=W.FileUpload(accept='.csv,.txt',multiple=False,description="Upload CSV")
LOAD_UPLOAD=W.Button(description="Load upload")
DATA_NOTE=W.HTML("<span style='color:#666'>Two columns. First column=time; second=mean u or specimen mass.</span>")

SYN_TIMES=W.Text(value="0,1,2,4,8,12,24",description="times",layout=W.Layout(width="430px"))
SYN_NOISE=W.FloatText(value=0.0,description="noise [%]")
SYN_PERTURB=W.FloatText(value=3.0,description="init factor")
SYN_DO_PERTURB=W.Checkbox(value=True,description="perturb estimated initial guesses after generation",indent=False)
GEN_SYN=W.Button(description="Generate from current truth",button_style="info",icon="flask")
LOAD_VERIFIED=W.Button(description="Load verified 1D example",icon="check")
SYN_NOTE=W.HTML()
SYN_STATE={"truth_unknowns":{},"truth_summary":"","generated":False}

RELRES=W.Checkbox(value=False,description="relative residuals",indent=False)
RFLOOR=W.FloatText(value=1e-6,description="relative floor")
MAXNFEV=W.IntText(value=60,description="max nfev")
XTOL=W.FloatText(value=1e-8,description="xtol"); FTOL=W.FloatText(value=1e-8,description="ftol"); GTOL=W.FloatText(value=1e-8,description="gtol")
IRUN=W.Button(description="Run inverse",button_style="warning",icon="cogs")
ISTATUS=W.HTML(); IOUT=W.Output(); IQUEST=W.Textarea(layout=W.Layout(width="100%",height="190px"))


# Dedicated sensitivity / identifiability panel
SENS_RANK_TOL=W.FloatText(value=1e-6,description="rank tol",layout=W.Layout(width="220px"))
SENS_AUTO=W.Checkbox(value=True,description="auto after inverse",indent=False)
SENS_RUN=W.Button(description="Analyze last fit",icon="bar-chart")
SENS_STATUS=W.HTML()
SENSOUT=W.Output()
LAST_INVERSE_RESULT={"result":None}


def update_inverse_dim(*_):
    d=IDIM.value
    ILy.layout.display=INy.layout.display="" if d==2 else "none"
    IBCBOX.children=tuple(IBCS[f].box for f in FACE_LABELS[d])
    update_inverse_D()


def update_inverse_D(*_):
    d=IDIM.value
    if IDSTRUCT.value=="Isotropic":
        IDW[0].kind.description="D(u)"; IDBOX.children=(IDW[0].box,)
    else:
        IDW[0].kind.description="D_x(u)"; IDW[1].kind.description="D_y(u)"; IDBOX.children=tuple(IDW[i].box for i in range(d))
    refresh_inverse_questions()


def data_mode_changed(*_):
    DRY_MASS.layout.display="" if (OBS_SOURCE.value=="Experimental" and DATA_MODE.value=="time, mass") else "none"


def observation_source_changed(*_):
    synthetic=OBS_SOURCE.value=="Synthetic"
    DATA_TEXT.disabled=synthetic
    UPLOAD.disabled=synthetic; LOAD_UPLOAD.disabled=synthetic; DATA_MODE.disabled=synthetic
    SYN_TIMES.disabled=not synthetic; SYN_NOISE.disabled=not synthetic; SYN_PERTURB.disabled=not synthetic
    SYN_DO_PERTURB.disabled=not synthetic; GEN_SYN.disabled=not synthetic; LOAD_VERIFIED.disabled=not synthetic
    if synthetic:
        DATA_NOTE.value=("<span style='color:#666'><b>Synthetic mode:</b> data are generated by the same forward engine. "
                         "Use this first to verify round-trip recovery before fitting experiments.</span>")
    else:
        DATA_NOTE.value=("<span style='color:#666'><b>Experimental mode:</b> two columns: time and mean u (or mass). "
                         "A successful optimizer termination is not automatically a good physical fit.</span>")
    data_mode_changed(); refresh_inverse_questions()


def load_upload(_):
    try:
        v=UPLOAD.value
        if not v: return
        if isinstance(v,dict): item=next(iter(v.values()))
        else: item=v[0]
        content=item["content"] if isinstance(item,dict) else item.content
        if hasattr(content,'tobytes'): content=content.tobytes()
        DATA_TEXT.value=bytes(content).decode('utf-8-sig')
    except Exception as e:
        ISTATUS.value=f"<span style='color:#b00020'>Upload read error: {e}</span>"
LOAD_UPLOAD.on_click(load_upload)


def collect_inverse_model_only():
    d=IDIM.value; lengths=[ILx.value,ILy.value]; cells=[INx.value,INy.value]
    if any(v<=0 for v in lengths[:d]): raise ValueError("All active lengths must be >0.")
    if any(v<2 for v in cells[:d]): raise ValueError("Each active mesh count must be >=2.")
    if IDT.value<=0: raise ValueError("dt max must be >0.")
    axis_specs=[IDW[i].spec() for i in range(1 if IDSTRUCT.value=="Isotropic" else d)]
    return dict(dim=d,lengths=lengths,cells=cells,u0=IU0.value,dt_max=IDT.value,D_structure=IDSTRUCT.value,
                axis_D=axis_specs,bcs={f:IBCS[f].spec() for f in FACE_LABELS[d]})


def collect_inverse():
    cfg=collect_inverse_model_only()
    mode="time, mean u" if OBS_SOURCE.value=="Synthetic" else DATA_MODE.value
    times,obs=parse_observations(DATA_TEXT.value,mode,TIME_UNIT.value,DRY_MASS.value)
    cfg.update(times=times,obs=obs,relative_residual=RELRES.value,residual_floor=RFLOOR.value,max_nfev=MAXNFEV.value,
               xtol=XTOL.value,ftol=FTOL.value,gtol=GTOL.value)
    return cfg


def _truth_unknown_map(cfg):
    truth={}; d=cfg["dim"]; n_axis=1 if cfg["D_structure"]=="Isotropic" else d
    for axis in range(n_axis):
        for pname,row in cfg["axis_D"][axis]["params"].items():
            if row["estimate"]: truth[f"D_{AXIS_NAMES[axis]}.{pname}"]=float(row["value"])
    for face,s in cfg["bcs"].items():
        if s["type"]=="Robin" and s["h"]["estimate"]: truth[f"{face}.h"]=float(s["h"]["value"])
    return truth


def _perturb_estimated_initial_guesses(factor):
    factor=float(factor)
    if factor<=0: raise ValueError("init factor must be >0.")
    d=IDIM.value; n_axis=1 if IDSTRUCT.value=="Isotropic" else d
    for w in IDW[:n_axis]:
        for r in w.rows.values():
            if r.mode.value!="Estimate": continue
            v=float(r.value.value); lo=float(r.lo.value); hi=float(r.hi.value)
            if r.log.value and v>0:
                nv=v*factor
            elif v!=0:
                nv=v*factor
            else:
                nv=lo+0.35*(hi-lo)
            eps=1e-9*max(abs(hi-lo),1.0)
            r.value.value=min(max(nv,lo+eps),hi-eps)
    for f in FACE_LABELS[d]:
        b=IBCS[f]
        if b.kind.value=="Robin" and b.h.mode.value=="Estimate":
            r=b.h; v=float(r.value.value); lo=float(r.lo.value); hi=float(r.hi.value)
            nv=v*factor if v!=0 else lo+0.35*(hi-lo)
            eps=1e-9*max(abs(hi-lo),1.0)
            r.value.value=min(max(nv,lo+eps),hi-eps)


def generate_synthetic_from_current(_=None):
    with IOUT:
        clear_output(wait=True)
        try:
            cfg=collect_inverse_model_only()
            display_times,times=parse_time_list(SYN_TIMES.value,TIME_UNIT.value)
            truth=_truth_unknown_map(cfg)
            pred=physical_predict_from_ui(cfg,times,store_fields=False)["mean_series"]
            noise_pct=float(SYN_NOISE.value)
            if noise_pct<0: raise ValueError("noise [%] must be >=0.")
            if noise_pct>0:
                rng=np.random.default_rng(12345)
                amp=max(float(np.ptp(pred)),0.1*max(float(np.max(np.abs(pred))),1e-12),1e-12)
                pred=pred+rng.normal(0.0,noise_pct/100.0*amp,size=pred.shape)
            DATA_TEXT.value="\n".join(f"{tt:.12g},{yy:.12g}" for tt,yy in zip(display_times,pred))
            SYN_STATE["truth_unknowns"]=truth; SYN_STATE["generated"]=True
            SYN_STATE["truth_summary"]="; ".join(f"{k}={v:.6g}" for k,v in truth.items()) or "No estimated parameters selected"
            if SYN_DO_PERTURB.value: _perturb_estimated_initial_guesses(SYN_PERTURB.value)
            SYN_NOTE.value=("<span style='color:green'><b>Synthetic data generated.</b> Truth for estimated parameters: "
                            +SYN_STATE["truth_summary"]+". Regenerate after changing geometry, BCs, or model form.</span>")
            print("Synthetic round-trip data generated with the current physical values.")
            print("Truth:",SYN_STATE["truth_summary"])
            if SYN_DO_PERTURB.value: print(f"Estimated-parameter initial guesses were perturbed by factor {SYN_PERTURB.value:g} where applicable.")
        except Exception as e:
            print(type(e).__name__+":",e)
            SYN_NOTE.value=f"<span style='color:#b00020'><b>Synthetic generation error:</b> {e}</span>"
GEN_SYN.on_click(generate_synthetic_from_current)


def load_verified_example(_=None):
    # Canonical 1D constant-D + fixed Robin-h round-trip test.
    IDIM.value=1; ILx.value=0.02; INx.value=30; IU0.value=0.30; IDT.value=1800.0
    IDSTRUCT.value="Isotropic"; IDW[0].kind.value="Constant"
    r=IDW[0].rows["a"]
    r.mode.value="Estimate"; r.value.value=1e-10; r.lo.value=1e-14; r.hi.value=1e-6; r.log.value=True
    for f in ["x-","x+"]:
        b=IBCS[f]; b.kind.value="Robin"; b.env.value=0.10
        b.h.mode.value="Fixed"; b.h.value.value=2e-8; b.h.lo.value=1e-12; b.h.hi.value=1e-4; b.h.log.value=True
    TIME_UNIT.value="h"; SYN_TIMES.value="0,1,2,4,8,12,24"; SYN_NOISE.value=0.0
    SYN_DO_PERTURB.value=True; SYN_PERTURB.value=3.0
    OBS_SOURCE.value="Synthetic"
    generate_synthetic_from_current()
    SYN_NOTE.value += "<br><b>Verified example:</b> D_true=1e-10 m²/s, h=2e-8 m/s fixed; inverse starts from D=3e-10."
LOAD_VERIFIED.on_click(load_verified_example)


def plot_inverse(r):
    t=r["base"]["times"]; obs=r["base"]["obs"]; pred=r["final"]["mean_series"]
    plt.figure(figsize=(8,4)); plt.plot(t,obs,'o',label='observed'); plt.plot(t,pred,'-',label='fitted')
    plt.xlabel('time [s]'); plt.ylabel('mean u'); plt.title('Inverse fit'); plt.legend(); plt.tight_layout(); plt.show()
    plt.figure(figsize=(8,3.5)); plt.axhline(0,lw=1); plt.plot(t,r["raw_residual"],'o-')
    plt.xlabel('time [s]'); plt.ylabel('prediction - observation'); plt.title('Residuals'); plt.tight_layout(); plt.show()

    print("\nEstimated parameter set")
    print("-"*72)
    vals=r["final"]["physical_unknowns"]
    truth=SYN_STATE.get("truth_unknowns",{}) if OBS_SOURCE.value=="Synthetic" else {}
    for u,v in zip(r["unknowns"],vals):
        line=f"{u['name']:<24s} = {v:.8g}" + ("   [log-space]" if u['log'] else "")
        if u['name'] in truth:
            tv=truth[u['name']]; rel=abs(v-tv)/max(abs(tv),1e-300)
            line += f"   true={tv:.8g}   rel.err={rel:.3e}"
        print(line)
    print("-"*72)
    o=r["opt"]; d=r["diagnostics"]
    print(f"optimizer success={o.success}; status={o.status}; nfev={o.nfev}; cost={o.cost:.6g}; optimality={o.optimality:.6g}")
    print(f"forward evaluations counted={r['evaluations']}")
    print(f"FIT QUALITY = {d['grade']}: RMSE={d['rmse']:.6g}; NRMSE(dynamic-range)={d['nrmse']:.3%}; R²={d['r2']:.6g}")
    if d['bound_flags']:
        for name,side,p in d['bound_flags']:
            print(f"WARNING: {name} is near its {side} optimization bound (normalized position={p:.4f}).")
    if d['systematic_same_sign']:
        print("WARNING: residuals are predominantly one sign — possible systematic model mismatch.")
    print(f"Jacobian numerical rank = {r['rank']} / {len(r['unknowns'])}")
    print(f"Jacobian condition indicator = {r['condition']:.6g}")
    print("singular values =",np.array2string(r["singular_values"],precision=4))
    if r['rank'] < len(r['unknowns']):
        print("WARNING: local rank deficiency — at least one parameter combination is not independently identifiable near this solution.")
    elif not np.isfinite(r['condition']) or r['condition']>1e6:
        print("WARNING: locally ill-conditioned parameter estimation; inspect parameter correlations / profile likelihood / alternative experiments.")
    print("NOTE: optimizer success and fit quality are separate diagnostics.")



def render_sensitivity_identifiability(r):
    d=compute_local_sensitivity_identifiability(r,SENS_RANK_TOL.value)
    names=[u["name"] for u in r["unknowns"]]
    t=d["times"]

    # 1. Time-resolved local normalized sensitivities
    plt.figure(figsize=(8,4.2))
    for j,name in enumerate(names):
        plt.plot(t,d["S"][:,j],marker='o',ms=3,label=name)
    plt.axhline(0,lw=1)
    plt.xlabel("time [s]")
    plt.ylabel("normalized local sensitivity")
    plt.title("Local sensitivity of mean response")
    if len(names)<=10: plt.legend()
    plt.tight_layout(); plt.show()

    # 2. RMS sensitivity strength
    plt.figure(figsize=(8,3.6))
    x=np.arange(len(names))
    plt.bar(x,d["rms"])
    plt.xticks(x,names,rotation=30,ha='right')
    plt.ylabel("RMS normalized sensitivity")
    plt.title("Relative local sensitivity strength")
    plt.tight_layout(); plt.show()

    # 3. Sensitivity-column collinearity
    if len(names)>=2:
        plt.figure(figsize=(6.2,5.2))
        im=plt.imshow(d["collinearity"],vmin=-1,vmax=1,aspect='auto')
        plt.colorbar(im,label="cosine similarity")
        plt.xticks(np.arange(len(names)),names,rotation=35,ha='right')
        plt.yticks(np.arange(len(names)),names)
        plt.title("Sensitivity collinearity matrix")
        plt.tight_layout(); plt.show()

    # 4. Singular-value spectrum
    if d["singular_values"].size:
        plt.figure(figsize=(7.5,3.6))
        k=np.arange(1,len(d["singular_values"])+1)
        plt.semilogy(k,d["relative_singular_values"],'o-')
        plt.axhline(d["rel_rank_tol"],ls='--',lw=1)
        plt.xlabel("singular-value index")
        plt.ylabel("relative singular value")
        plt.title("Normalized sensitivity singular-value spectrum")
        plt.tight_layout(); plt.show()

    print("\nSensitivity / identifiability diagnostics")
    print("-"*84)
    print("Definition: log-space parameter -> (1/y_scale) d(mean u)/d ln(theta)")
    print("            linear-space parameter -> parameter-scaled derivative / y_scale")
    print(f"response normalization y_scale = {d['yscale']:.6g}")
    print(f"effective rank = {d['effective_rank']} / {len(names)}   (relative SVD tolerance={d['rel_rank_tol']:.1e})")
    print(f"normalized sensitivity condition indicator = {d['condition']:.6g}")
    if len(names)>=2:
        print(f"maximum |sensitivity collinearity| = {d['max_abs_collinearity']:.6g}")
    print(f"LOCAL IDENTIFIABILITY = {d['grade']}: {d['reason']}")
    print("\nParameter sensitivity strength")
    for j,name in enumerate(names):
        print(f"  {name:<24s} RMS={d['rms'][j]:.6e}   relative={d['relative_strength'][j]:.3%}   {d['sensitivity_coordinates'][j]}")
    if d["low_sensitivity"]:
        print("WARNING: very low local sensitivity relative to the strongest parameter:",", ".join(d["low_sensitivity"]))
    if d["high_collinearity"]:
        print("WARNING: highly similar sensitivity histories (possible parameter compensation):")
        for a,b,c in d["high_collinearity"]:
            print(f"  {a} vs {b}: cosine similarity={c:.6g}")
    if len(names)>=2 and d["effective_rank"]<len(names):
        print("WARNING: effective local rank deficiency under the selected relative SVD tolerance.")
    print("NOTE: This panel diagnoses LOCAL practical identifiability near the fitted solution; it does not prove global uniqueness.")
    return d


def analyze_last_fit(_=None):
    r=LAST_INVERSE_RESULT.get("result")
    with SENSOUT:
        clear_output(wait=True)
        if r is None:
            print("Run inverse first; no fitted solution is available for sensitivity analysis.")
            SENS_STATUS.value="<span style='color:#b36b00'><b>No inverse result yet.</b></span>"
            return
        try:
            d=render_sensitivity_identifiability(r)
            color={"GOOD":"green","REVIEW":"#b36b00","POOR":"#b00020"}.get(d["grade"],"#333")
            SENS_STATUS.value=f"<span style='color:{color}'><b>Local identifiability {d['grade']}.</b></span>"
        except Exception as e:
            print(type(e).__name__+":",e)
            SENS_STATUS.value=f"<span style='color:#b00020'><b>Sensitivity error:</b> {e}</span>"
SENS_RUN.on_click(analyze_last_fit)


def refresh_inverse_questions(*_):
    d=IDIM.value; st=IDSTRUCT.value
    n_est=0
    for w in IDW[:1 if st=="Isotropic" else d]: n_est+=sum(r.mode.value=="Estimate" for r in w.rows.values())
    for f in FACE_LABELS[d]:
        if IBCS[f].kind.value=="Robin" and IBCS[f].h.mode.value=="Estimate": n_est+=1
    qs=["Can this inverse setup recover known parameters from synthetic data before I trust an experimental fit?",
        f"Are {n_est} estimated parameters supportable from only the mean-response time curve?",
        "Can diffusivity and Robin surface-transfer coefficients compensate for one another and produce nearly identical fits?",
        "If an estimated parameter reaches a bound, does that indicate inadequate bounds, model mismatch, or low sensitivity?",
        "Does optimizer success mean the fit is physically adequate?",
        "What do the optimizer Jacobian singular values say about local identifiability?"]
    if d==2 and st=="Diagonal anisotropic":
        qs.append("Can mean data alone distinguish D_x(u) from D_y(u), or does the specimen aspect ratio / boundary exposure need directional information?")
    qs.append("How should I design additional specimen dimensions, boundary conditions, or measurements to improve identifiability?")
    IQUEST.value="\n\n".join("• "+q for q in qs)


def run_inverse_clicked(_):
    ISTATUS.value="<b>Running inverse...</b>"
    with IOUT:
        clear_output(wait=True)
        try:
            if OBS_SOURCE.value=="Synthetic" and not DATA_TEXT.value.strip():
                raise ValueError("Generate synthetic data first (or load the verified example).")
            cfg=collect_inverse(); t0=time.perf_counter(); r=run_inverse(cfg); LAST_INVERSE_RESULT["result"]=r; plot_inverse(r)
            if SENS_AUTO.value: analyze_last_fit()
            print(f"Wall time: {time.perf_counter()-t0:.3f} s")
            d=r["diagnostics"]
            ISTATUS.value=f"<span style='color:{d['color']}'><b>Optimization completed — fit {d['grade']}.</b></span>"
        except Exception as e:
            print(type(e).__name__+":",e); ISTATUS.value=f"<span style='color:#b00020'><b>Error:</b> {e}</span>"

IRUN.on_click(run_inverse_clicked)
IDIM.observe(update_inverse_dim,names="value"); IDSTRUCT.observe(update_inverse_D,names="value"); DATA_MODE.observe(data_mode_changed,names="value")
OBS_SOURCE.observe(observation_source_changed,names="value")
for w in IDW:
    w.kind.observe(refresh_inverse_questions,names="value")
    for r in w.rows.values(): r.mode.observe(refresh_inverse_questions,names="value")
for b in IBCS.values():
    b.kind.observe(refresh_inverse_questions,names="value"); b.h.mode.observe(refresh_inverse_questions,names="value")
update_inverse_dim(); update_inverse_D(); observation_source_changed(); refresh_inverse_questions()

SYN_PANEL=W.VBox([W.HTML("<b>Round-trip verification</b>"),W.HBox([SYN_TIMES,TIME_UNIT]),
                  W.HBox([SYN_NOISE,SYN_PERTURB]),SYN_DO_PERTURB,W.HBox([LOAD_VERIFIED,GEN_SYN]),SYN_NOTE])
EXP_PANEL=W.VBox([W.HBox([DATA_MODE,TIME_UNIT,DRY_MASS]),W.HBox([UPLOAD,LOAD_UPLOAD])])
DATA_PANEL=W.VBox([OBS_SOURCE,SYN_PANEL,EXP_PANEL,DATA_TEXT,DATA_NOTE])

def _toggle_obs_panels(*_):
    SYN_PANEL.layout.display="" if OBS_SOURCE.value=="Synthetic" else "none"
    EXP_PANEL.layout.display="" if OBS_SOURCE.value=="Experimental" else "none"
observation_source_changed(); _toggle_obs_panels()
OBS_SOURCE.observe(_toggle_obs_panels,names="value")

IGEOM=W.VBox([W.HBox([ILx,INx]),W.HBox([ILy,INy]),W.HBox([IU0,IDT])])
IOPT=W.VBox([W.HBox([RELRES,RFLOOR]),W.HBox([MAXNFEV,XTOL,FTOL,GTOL]),
             W.HTML("<span style='color:#666'>Optimizer convergence is reported separately from fit quality, bound proximity, and local identifiability diagnostics.</span>")])
SENS_PANEL=W.VBox([
    W.HTML("<b>Local sensitivity / identifiability</b><br><span style='color:#666'>Uses the final least-squares Jacobian. Log-space parameters are analyzed as d(mean u)/d ln(theta). This is a local practical-identifiability diagnostic, not proof of global uniqueness.</span>"),
    W.HBox([SENS_RANK_TOL,SENS_AUTO,SENS_RUN,SENS_STATUS]),
    SENSOUT,
])
IACC=W.Accordion(children=[DATA_PANEL,IGEOM,W.VBox([IDSTRUCT,IDBOX]),IBCBOX,IOPT,SENS_PANEL])
for i,t in enumerate(["Observation data / round-trip test","Geometry / time stepping","Diffusivity + Fixed/Estimate","Boundary conditions + h estimation","Optimizer / fit diagnostics","Sensitivity / identifiability"]): IACC.set_title(i,t)
IACC.selected_index=0
INVERSE_PANEL=W.VBox([W.HTML("<h3>Inverse: 1D / 2D FVM</h3>"),IDIM,IACC,W.HBox([IRUN,ISTATUS]),IOUT,
                      W.HTML("<h3>What should I ask AI?</h3>"),IQUEST])

# Populate a physically self-consistent default dataset at startup.
load_verified_example()


In [ ]:
# ============================================================
# v2.3 — Input / Output / Save / Load / Reproducibility layer
# ============================================================
import json, csv, re, html
from datetime import datetime
from IPython.display import HTML

APP_VERSION = "v2.3"

LAST_FORWARD_RESULT = {
    "result": None,
    "config": None,
    "ui_state": None,
    "elapsed_s": None,
    "run_started_at": None,
}
LAST_INVERSE_RESULT.update({
    "config": None,
    "ui_state": None,
    "elapsed_s": None,
    "run_started_at": None,
    "sensitivity": None,
})


# -----------------------------
# Generic serialization helpers
# -----------------------------
def _jsonable(obj):
    if isinstance(obj, dict):
        return {str(k): _jsonable(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple)):
        return [_jsonable(v) for v in obj]
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    if isinstance(obj, np.generic):
        return obj.item()
    if isinstance(obj, Path):
        return str(obj)
    if isinstance(obj, (str, int, float, bool)) or obj is None:
        return obj
    return str(obj)


def _fmt(v):
    if v is None:
        return "—"
    if isinstance(v, (bool, np.bool_)):
        return "Yes" if bool(v) else "No"
    if isinstance(v, (int, np.integer)):
        return str(int(v))
    if isinstance(v, (float, np.floating)):
        if not np.isfinite(v):
            return str(v)
        av = abs(float(v))
        if av != 0 and (av < 1e-4 or av >= 1e5):
            return f"{float(v):.6e}"
        return f"{float(v):.6g}"
    return str(v)


def _table_html(title, headers, rows):
    title_html = f"<h4 style='margin:8px 0 4px 0'>{html.escape(str(title))}</h4>" if title else ""
    h = "".join(
        f"<th style='text-align:left;padding:4px 8px;border-bottom:1px solid #aaa'>{html.escape(str(x))}</th>"
        for x in headers
    )
    body = []
    for row in rows:
        body.append("<tr>" + "".join(
            f"<td style='padding:3px 8px;border-bottom:1px solid #eee;vertical-align:top'>{html.escape(_fmt(x))}</td>"
            for x in row
        ) + "</tr>")
    return (
        title_html
        + "<div style='overflow-x:auto'><table style='border-collapse:collapse;font-size:13px'>"
        + f"<thead><tr>{h}</tr></thead><tbody>{''.join(body)}</tbody></table></div>"
    )


def _show_html(out_widget, chunks):
    with out_widget:
        clear_output(wait=True)
        display(HTML("".join(chunks)))


def _flatten_dict(obj, prefix="", rows=None):
    if rows is None:
        rows = []
    if isinstance(obj, dict):
        for k, v in obj.items():
            p = f"{prefix}.{k}" if prefix else str(k)
            _flatten_dict(v, p, rows)
    elif isinstance(obj, (list, tuple)):
        for i, v in enumerate(obj):
            _flatten_dict(v, f"{prefix}[{i}]", rows)
    else:
        rows.append((prefix, _jsonable(obj)))
    return rows


def _write_csv(path, headers, rows):
    path = Path(path)
    with path.open("w", newline="", encoding="utf-8-sig") as f:
        w = csv.writer(f)
        w.writerow(headers)
        for row in rows:
            w.writerow([_jsonable(v) for v in row])


def _safe_label(s):
    s = re.sub(r"[^A-Za-z0-9._-]+", "_", str(s).strip())
    return s.strip("_")[:48]


def _make_run_dir(base_dir, mode, label=""):
    base = Path(base_dir).expanduser()
    base.mkdir(parents=True, exist_ok=True)
    stamp = datetime.now().astimezone().strftime("%Y%m%d_%H%M%S")
    suffix = _safe_label(label)
    name = f"{stamp}_{mode.lower()}" + (f"_{suffix}" if suffix else "")
    p = base / name
    k = 1
    while p.exists():
        p = base / f"{name}_{k:02d}"
        k += 1
    p.mkdir()
    return p


def _saved_run_choices(base_dir, mode):
    base = Path(base_dir).expanduser()
    if not base.exists():
        return []
    found = []
    for p in sorted(base.iterdir(), reverse=True):
        f = p / "input.json"
        if not p.is_dir() or not f.exists():
            continue
        try:
            rec = json.loads(f.read_text(encoding="utf-8"))
            if str(rec.get("mode", "")).lower() == mode.lower():
                found.append((p.name, str(p)))
        except Exception:
            pass
    return found


def _physical_value(encoded_spec, z):
    return float(np.exp(z)) if encoded_spec.get("log", False) else float(z)


# -----------------------------
# Forward state + summaries
# -----------------------------
def serialize_forward_ui():
    return _jsonable(collect_forward())


def restore_forward_ui(state):
    METHOD.disabled = False
    DIM.value = int(state["dim"])
    lengths = state["lengths"]
    cells = state["cells"]
    Lx.value = float(lengths[0]); Ly.value = float(lengths[1]); Lz.value = float(lengths[2])
    Nx.value = int(cells[0]); Ny.value = int(cells[1]); Nz.value = int(cells[2])

    IC_KIND.value = state["IC_kind"]
    p = state["IC_params"]
    IC_VALUE.value = float(p["value"])
    IC_LEFT.value = float(p["left"]); IC_RIGHT.value = float(p["right"])
    IC_BASE.value = float(p["base"]); IC_AMP.value = float(p["amplitude"])
    IC_SIGMA.value = float(p["sigma"]); IC_SPLIT.value = float(p["split"])
    IC_EXPR.value = str(p["expression"])

    D_STRUCTURE.value = state["D_structure"]
    for i, spec in enumerate(state["axis_D"]):
        w = FD[i]
        w.kind.value = spec["kind"]
        pp = spec["params"]
        w.a.value = float(pp["a"]); w.b.value = float(pp["b"]); w.c.value = float(pp["c"])
        w.ur.value = float(pp["u_ref"])
        w.c0.value = float(pp["c0"]); w.c1.value = float(pp["c1"]); w.c2.value = float(pp["c2"])
        w.expr.value = str(pp["expression"])

    for face, spec in state["bcs"].items():
        w = FACES[face]
        w.kind.value = spec["type"]
        w.value.value = float(spec["value"])
        w.flux.value = float(spec["flux"])
        w.h.value = float(spec["h"])
        w.env.value = float(spec["env"])

    DT.value = float(state["dt"])
    NSTEPS.value = int(state["nsteps"])
    NONLINEAR.value = state["nonlinear_method"]
    METHOD.value = state["method"]
    update_forward_geometry()
    update_forward_ic()
    update_forward_D()
    refresh_forward_questions()


def _forward_input_chunks(cfg):
    d = int(cfg["dim"])
    chunks = [
        _table_html("Run setup", ["Item", "Value"], [
            ["App version", APP_VERSION],
            ["Method", cfg["method"]],
            ["Dimension", f"{d}D"],
            ["D structure", cfg["D_structure"]],
            ["Lengths [m]", cfg["lengths"][:d]],
            ["Mesh counts", cfg["cells"][:d]],
            ["dt [s]", cfg["dt"]],
            ["Steps", cfg["nsteps"]],
            ["Final nominal time [s]", cfg["dt"] * cfg["nsteps"]],
            ["Nonlinear iteration", cfg["nonlinear_method"]],
        ])
    ]
    icp = cfg["IC_params"]
    chunks.append(_table_html("Initial condition", ["Item", "Value"], [
        ["Type", cfg["IC_kind"]],
        ["Parameters", json.dumps(_jsonable(icp), ensure_ascii=False)],
    ]))
    drows = []
    for i, spec in enumerate(cfg["axis_D"]):
        axis = "all" if cfg["D_structure"] == "Isotropic" else AXIS_NAMES[i]
        drows.append([
            axis,
            spec["kind"],
            json.dumps(_jsonable(spec["params"]), ensure_ascii=False),
        ])
    chunks.append(_table_html("Diffusivity", ["Direction", "Model", "Parameters"], drows))
    brows = []
    for face, spec in cfg["bcs"].items():
        if spec["type"] == "Robin":
            detail = f"h={_fmt(spec['h'])}; env={_fmt(spec['env'])}"
        elif spec["type"] == "Dirichlet":
            detail = f"value={_fmt(spec['value'])}"
        else:
            detail = f"flux={_fmt(spec['flux'])}"
        brows.append([face, spec["type"], detail])
    chunks.append(_table_html("Boundary conditions", ["Face", "Type", "Values"], brows))
    return chunks


def render_forward_input_summary(cfg):
    _show_html(FINPUT_SUMMARY, _forward_input_chunks(cfg))


def _forward_summary_dict(res, elapsed):
    U = np.asarray(res["final"])
    c = np.asarray(res.get("iteration_counts", []), dtype=float)
    out = {
        "app_version": APP_VERSION,
        "method": res["method"],
        "dimension": int(res["dim"]),
        "D_structure": res["D_structure"],
        "final_min": float(np.min(U)),
        "final_max": float(np.max(U)),
        "final_mean": float(res["mean_series"][-1]),
        "elapsed_s": float(elapsed),
        "mean_nonlinear_iterations": float(np.mean(c)) if c.size else np.nan,
        "max_nonlinear_iterations": float(np.max(c)) if c.size else np.nan,
    }
    try:
        _, q = structured_flux(res)
        if q is not None:
            qmag = np.sqrt(sum(np.asarray(qi, dtype=float)**2 for qi in q))
            out["flux_mag_min"] = float(np.nanmin(qmag))
            out["flux_mag_max"] = float(np.nanmax(qmag))
            out["flux_mag_mean"] = float(np.nanmean(qmag))
    except Exception:
        pass
    return out


def render_forward_result_summary(res, elapsed):
    s = _forward_summary_dict(res, elapsed)
    rows = [[k, v] for k, v in s.items()]
    chunks = [_table_html("Forward result summary", ["Metric", "Value"], rows)]
    chunks.append(_table_html("Constitutive expressions", ["Expression"], [[x] for x in res.get("D_formulas", [])]))
    _show_html(FRESULT_SUMMARY, chunks)

    times = np.asarray(res["times"], dtype=float)
    means = np.asarray(res["mean_series"], dtype=float)
    counts = np.asarray(res.get("iteration_counts", []))
    trows = []
    for i, (tt, mm) in enumerate(zip(times, means)):
        nit = "" if i == 0 or counts.size < i else counts[i-1]
        trows.append([i, tt, mm, nit])
    _show_html(FDETAIL_SUMMARY, [
        _table_html("Forward time series", ["Index", "time [s]", "mean u", "nonlinear iterations"], trows)
    ])


# -----------------------------
# Inverse state + summaries
# -----------------------------
def serialize_inverse_ui():
    model = collect_inverse_model_only()
    return _jsonable({
        "model": model,
        "observation": {
            "source": OBS_SOURCE.value,
            "data_mode": DATA_MODE.value,
            "time_unit": TIME_UNIT.value,
            "dry_mass": DRY_MASS.value,
            "data_text": DATA_TEXT.value,
            "synthetic_times": SYN_TIMES.value,
            "synthetic_noise_pct": SYN_NOISE.value,
            "synthetic_init_factor": SYN_PERTURB.value,
            "synthetic_perturb": SYN_DO_PERTURB.value,
            "synthetic_state": SYN_STATE,
        },
        "optimizer": {
            "relative_residuals": RELRES.value,
            "relative_floor": RFLOOR.value,
            "max_nfev": MAXNFEV.value,
            "xtol": XTOL.value,
            "ftol": FTOL.value,
            "gtol": GTOL.value,
        },
        "sensitivity": {
            "rank_tol": SENS_RANK_TOL.value,
            "auto_after_inverse": SENS_AUTO.value,
        },
    })


def _restore_inverse_parameter(row, spec):
    row.mode.value = "Estimate" if spec["estimate"] else "Fixed"
    row.value.value = float(spec["value"])
    row.lo.value = float(spec["lower"])
    row.hi.value = float(spec["upper"])
    row.log.value = bool(spec["log"])


def restore_inverse_ui(state):
    model = state["model"]
    d = int(model["dim"])
    IDIM.value = d
    ILx.value = float(model["lengths"][0]); ILy.value = float(model["lengths"][1])
    INx.value = int(model["cells"][0]); INy.value = int(model["cells"][1])
    IU0.value = float(model["u0"]); IDT.value = float(model["dt_max"])
    IDSTRUCT.value = model["D_structure"]

    for i, spec in enumerate(model["axis_D"]):
        w = IDW[i]
        w.kind.value = spec["kind"]
        for pname, pspec in spec["params"].items():
            _restore_inverse_parameter(w.rows[pname], pspec)

    for face, spec in model["bcs"].items():
        w = IBCS[face]
        w.kind.value = spec["type"]
        w.dir_value.value = float(spec["value"])
        w.flux.value = float(spec["flux"])
        w.env.value = float(spec["env"])
        _restore_inverse_parameter(w.h, spec["h"])

    obs = state["observation"]
    OBS_SOURCE.value = obs["source"]
    DATA_MODE.value = obs["data_mode"]
    TIME_UNIT.value = obs["time_unit"]
    DRY_MASS.value = float(obs["dry_mass"])
    SYN_TIMES.value = str(obs["synthetic_times"])
    SYN_NOISE.value = float(obs["synthetic_noise_pct"])
    SYN_PERTURB.value = float(obs["synthetic_init_factor"])
    SYN_DO_PERTURB.value = bool(obs["synthetic_perturb"])
    DATA_TEXT.value = str(obs["data_text"])
    SYN_STATE.clear()
    SYN_STATE.update(obs.get("synthetic_state", {
        "truth_unknowns": {}, "truth_summary": "", "generated": False
    }))

    op = state["optimizer"]
    RELRES.value = bool(op["relative_residuals"])
    RFLOOR.value = float(op["relative_floor"])
    MAXNFEV.value = int(op["max_nfev"])
    XTOL.value = float(op["xtol"]); FTOL.value = float(op["ftol"]); GTOL.value = float(op["gtol"])

    sens = state.get("sensitivity", {})
    if "rank_tol" in sens:
        SENS_RANK_TOL.value = float(sens["rank_tol"])
    if "auto_after_inverse" in sens:
        SENS_AUTO.value = bool(sens["auto_after_inverse"])

    update_inverse_dim()
    update_inverse_D()
    observation_source_changed()
    _toggle_obs_panels()
    refresh_inverse_questions()


def _inverse_input_chunks(cfg, ui_state):
    d = int(cfg["dim"])
    obs_state = ui_state["observation"]
    chunks = [
        _table_html("Run setup", ["Item", "Value"], [
            ["App version", APP_VERSION],
            ["Dimension", f"{d}D"],
            ["Method", "FVM"],
            ["D structure", cfg["D_structure"]],
            ["Lengths [m]", cfg["lengths"][:d]],
            ["Mesh counts", cfg["cells"][:d]],
            ["Initial u", cfg["u0"]],
            ["dt max [s]", cfg["dt_max"]],
            ["Observation source", obs_state["source"]],
            ["Data mode", obs_state["data_mode"]],
            ["Input time unit", obs_state["time_unit"]],
            ["Number of observations", len(cfg["times"])],
            ["Observation time range [s]", [float(cfg["times"][0]), float(cfg["times"][-1])]],
            ["Relative residuals", cfg["relative_residual"]],
        ])
    ]

    prows = []
    n_axis = 1 if cfg["D_structure"] == "Isotropic" else d
    for i, spec in enumerate(cfg["axis_D"][:n_axis]):
        axis = "all" if cfg["D_structure"] == "Isotropic" else AXIS_NAMES[i]
        for pname, pspec in spec["params"].items():
            prows.append([
                f"D_{axis}.{pname}",
                "Estimate" if pspec["estimate"] else "Fixed",
                pspec["value"],
                pspec["lower"] if pspec["estimate"] else "",
                pspec["upper"] if pspec["estimate"] else "",
                pspec["log"] if pspec["estimate"] else "",
            ])
    for face, spec in cfg["bcs"].items():
        if spec["type"] == "Robin":
            hs = spec["h"]
            prows.append([
                f"{face}.h",
                "Estimate" if hs["estimate"] else "Fixed",
                hs["value"],
                hs["lower"] if hs["estimate"] else "",
                hs["upper"] if hs["estimate"] else "",
                hs["log"] if hs["estimate"] else "",
            ])
    chunks.append(_table_html(
        "Parameter roles",
        ["Parameter", "Role", "Value / initial", "Lower", "Upper", "Log space"],
        prows
    ))

    brows = []
    for face, spec in cfg["bcs"].items():
        if spec["type"] == "Robin":
            detail = f"h={_fmt(spec['h']['value'])}; env={_fmt(spec['env'])}"
        elif spec["type"] == "Dirichlet":
            detail = f"value={_fmt(spec['value'])}"
        else:
            detail = f"flux={_fmt(spec['flux'])}"
        brows.append([face, spec["type"], detail])
    chunks.append(_table_html("Boundary conditions", ["Face", "Type", "Values"], brows))
    return chunks


def render_inverse_input_summary(cfg, ui_state):
    _show_html(IINPUT_SUMMARY, _inverse_input_chunks(cfg, ui_state))


def _inverse_summary_dict(r, elapsed, sens):
    d = r["diagnostics"]
    o = r["opt"]
    return {
        "app_version": APP_VERSION,
        "optimizer_success": bool(o.success),
        "optimizer_status": int(o.status),
        "nfev": int(o.nfev),
        "cost": float(o.cost),
        "optimality": float(o.optimality),
        "forward_evaluations": int(r["evaluations"]),
        "fit_grade": d["grade"],
        "rmse": float(d["rmse"]),
        "mae": float(d["mae"]),
        "nrmse": float(d["nrmse"]),
        "r2": float(d["r2"]) if np.isfinite(d["r2"]) else np.nan,
        "optimizer_jacobian_rank": int(r["rank"]),
        "optimizer_jacobian_condition": float(r["condition"]),
        "identifiability_grade": sens["grade"],
        "identifiability_effective_rank": int(sens["effective_rank"]),
        "identifiability_condition": float(sens["condition"]),
        "max_abs_sensitivity_collinearity": float(sens["max_abs_collinearity"]),
        "elapsed_s": float(elapsed),
    }


def _inverse_parameter_rows(r):
    vals = np.asarray(r["final"]["physical_unknowns"], dtype=float)
    saved_ui = LAST_INVERSE_RESULT.get("ui_state") or {}
    saved_obs = saved_ui.get("observation", {})
    saved_syn = saved_obs.get("synthetic_state", {})
    truth = saved_syn.get("truth_unknowns", {}) if saved_obs.get("source") == "Synthetic" else {}
    rows = []
    for u, zfit, v in zip(r["unknowns"], r["opt"].x, vals):
        initial = _physical_value(u, u["z0"])
        lower = _physical_value(u, u["lo"])
        upper = _physical_value(u, u["hi"])
        tv = truth.get(u["name"], None)
        rel = None if tv is None else abs(float(v)-float(tv))/max(abs(float(tv)), 1e-300)
        near = ""
        for name, side, p in r["diagnostics"]["bound_flags"]:
            if name == u["name"]:
                near = f"{side} ({p:.4f})"
        rows.append([
            u["name"], initial, float(v), lower, upper, bool(u["log"]),
            tv if tv is not None else "", rel if rel is not None else "", near,
        ])
    return rows


def render_inverse_result_summary(r, elapsed):
    sens = compute_local_sensitivity_identifiability(r, SENS_RANK_TOL.value)
    LAST_INVERSE_RESULT["sensitivity"] = sens
    summary = _inverse_summary_dict(r, elapsed, sens)
    chunks = [_table_html("Inverse result summary", ["Metric", "Value"], [[k, v] for k, v in summary.items()])]
    chunks.append(_table_html(
        "Estimated parameters",
        ["Parameter", "Initial", "Estimate", "Lower", "Upper", "Log", "Truth", "Rel. error", "Near bound"],
        _inverse_parameter_rows(r)
    ))
    _show_html(IRESULT_SUMMARY, chunks)

    t = np.asarray(r["base"]["times"], dtype=float)
    obs = np.asarray(r["base"]["obs"], dtype=float)
    pred = np.asarray(r["final"]["mean_series"], dtype=float)
    raw = np.asarray(r["raw_residual"], dtype=float)
    rows = [[i, tt, oo, pp, rr] for i, (tt, oo, pp, rr) in enumerate(zip(t, obs, pred, raw))]
    _show_html(IDETAIL_SUMMARY, [
        _table_html("Inverse time series", ["Index", "time [s]", "observed", "predicted", "residual"], rows)
    ])
    return sens


# -----------------------------
# Save functions
# -----------------------------
def _save_input_files(run_dir, record):
    (run_dir / "input.json").write_text(
        json.dumps(_jsonable(record), indent=2, ensure_ascii=False),
        encoding="utf-8",
    )
    flat = _flatten_dict(record["ui_state"])
    _write_csv(run_dir / "input.csv", ["key", "value"], flat)


def _save_forward_npz(run_dir, res):
    arrays = {
        "final": np.asarray(res["final"]),
        "times": np.asarray(res["times"]),
        "mean_series": np.asarray(res["mean_series"]),
        "iteration_counts": np.asarray(res.get("iteration_counts", [])),
    }
    if "axes_common" in res:
        for i, ax in enumerate(res["axes_common"]):
            arrays[f"axis_{AXIS_NAMES[i]}"] = np.asarray(ax)
    if "points" in res:
        arrays["points"] = np.asarray(res["points"])
    if "elems" in res:
        arrays["elems"] = np.asarray(res["elems"])
    np.savez_compressed(run_dir / "field.npz", **arrays)


def save_forward_run(_=None):
    rec = LAST_FORWARD_RESULT
    res = rec.get("result")
    if res is None:
        FSAVE_STATUS.value = "<span style='color:#b36b00'><b>No completed forward run to save.</b></span>"
        return None
    try:
        run_dir = _make_run_dir(FRUN_DIR.value, "forward", FRUN_LABEL.value)
        summary = _forward_summary_dict(res, rec["elapsed_s"])
        record = {
            "app_version": APP_VERSION,
            "mode": "Forward",
            "saved_at": datetime.now().astimezone().isoformat(timespec="seconds"),
            "run_started_at": rec["run_started_at"],
            "ui_state": rec["ui_state"],
            "summary": summary,
        }
        _save_input_files(run_dir, record)
        _write_csv(run_dir / "summary.csv", ["metric", "value"], list(summary.items()))

        times = np.asarray(res["times"], dtype=float)
        means = np.asarray(res["mean_series"], dtype=float)
        counts = np.asarray(res.get("iteration_counts", []))
        rows = []
        for i, (tt, mm) in enumerate(zip(times, means)):
            nit = "" if i == 0 or counts.size < i else int(counts[i-1])
            rows.append([i, tt, mm, nit])
        _write_csv(run_dir / "timeseries.csv", ["index", "time_s", "mean_u", "nonlinear_iterations"], rows)

        if FSAVE_FIELD.value:
            _save_forward_npz(run_dir, res)

        report = [
            f"# Nonlinear Diffusion Forward Run — {APP_VERSION}",
            "",
            f"- Saved: {record['saved_at']}",
            f"- Method: {summary['method']}",
            f"- Dimension: {summary['dimension']}D",
            f"- D structure: {summary['D_structure']}",
            f"- Final mean: {_fmt(summary['final_mean'])}",
            f"- Final min/max: {_fmt(summary['final_min'])} / {_fmt(summary['final_max'])}",
            f"- Wall time [s]: {_fmt(summary['elapsed_s'])}",
            "",
            "Input values are stored in `input.json` and `input.csv`.",
            "Time history is stored in `timeseries.csv`.",
        ]
        if FSAVE_FIELD.value:
            report.append("Final field and coordinates are stored in `field.npz`.")
        (run_dir / "report.md").write_text("\n".join(report), encoding="utf-8")

        refresh_forward_runs()
        FSAVE_STATUS.value = f"<span style='color:green'><b>Saved:</b> {html.escape(str(run_dir))}</span>"
        return run_dir
    except Exception as e:
        FSAVE_STATUS.value = f"<span style='color:#b00020'><b>Save error:</b> {html.escape(str(e))}</span>"
        return None


def _save_inverse_npz(run_dir, r):
    arrays = {
        "times": np.asarray(r["base"]["times"]),
        "observed": np.asarray(r["base"]["obs"]),
        "predicted": np.asarray(r["final"]["mean_series"]),
        "residual": np.asarray(r["raw_residual"]),
        "final": np.asarray(r["final"]["final"]),
    }
    fields = r["final"].get("fields", None)
    if fields is not None:
        try:
            arrays["fields_at_observation_times"] = np.asarray(fields)
        except Exception:
            pass
    for i, ax in enumerate(r["final"].get("axes", [])):
        arrays[f"axis_{AXIS_NAMES[i]}"] = np.asarray(ax)
    np.savez_compressed(run_dir / "field.npz", **arrays)


def save_inverse_run(_=None):
    rec = LAST_INVERSE_RESULT
    r = rec.get("result")
    if r is None:
        ISAVE_STATUS.value = "<span style='color:#b36b00'><b>No completed inverse run to save.</b></span>"
        return None
    try:
        sens = rec.get("sensitivity")
        if sens is None:
            sens = compute_local_sensitivity_identifiability(r, SENS_RANK_TOL.value)
            rec["sensitivity"] = sens

        run_dir = _make_run_dir(IRUN_DIR.value, "inverse", IRUN_LABEL.value)
        summary = _inverse_summary_dict(r, rec["elapsed_s"], sens)
        record = {
            "app_version": APP_VERSION,
            "mode": "Inverse",
            "saved_at": datetime.now().astimezone().isoformat(timespec="seconds"),
            "run_started_at": rec["run_started_at"],
            "ui_state": rec["ui_state"],
            "summary": summary,
        }
        _save_input_files(run_dir, record)
        _write_csv(run_dir / "summary.csv", ["metric", "value"], list(summary.items()))

        prows = _inverse_parameter_rows(r)
        _write_csv(
            run_dir / "parameters.csv",
            ["parameter", "initial", "estimate", "lower", "upper", "log_space", "truth", "relative_error", "near_bound"],
            prows,
        )

        t = np.asarray(r["base"]["times"], dtype=float)
        obs = np.asarray(r["base"]["obs"], dtype=float)
        pred = np.asarray(r["final"]["mean_series"], dtype=float)
        raw = np.asarray(r["raw_residual"], dtype=float)
        _write_csv(
            run_dir / "timeseries.csv",
            ["index", "time_s", "observed", "predicted", "residual"],
            [[i, tt, oo, pp, rr] for i, (tt, oo, pp, rr) in enumerate(zip(t, obs, pred, raw))],
        )

        diag_record = {
            "fit": r["diagnostics"],
            "optimizer_singular_values": r["singular_values"],
            "optimizer_rank": r["rank"],
            "optimizer_condition": r["condition"],
            "sensitivity_identifiability": sens,
        }
        (run_dir / "diagnostics.json").write_text(
            json.dumps(_jsonable(diag_record), indent=2, ensure_ascii=False),
            encoding="utf-8",
        )

        names = [u["name"] for u in r["unknowns"]]
        srows = []
        for i, tt in enumerate(np.asarray(sens["times"])):
            srows.append([tt] + list(np.asarray(sens["S"])[i, :]))
        _write_csv(run_dir / "sensitivity.csv", ["time_s"] + names, srows)

        if ISAVE_FIELD.value:
            _save_inverse_npz(run_dir, r)

        report = [
            f"# Nonlinear Diffusion Inverse Run — {APP_VERSION}",
            "",
            f"- Saved: {record['saved_at']}",
            f"- Fit grade: {summary['fit_grade']}",
            f"- RMSE: {_fmt(summary['rmse'])}",
            f"- NRMSE: {_fmt(summary['nrmse'])}",
            f"- R2: {_fmt(summary['r2'])}",
            f"- Local identifiability: {summary['identifiability_grade']}",
            f"- Effective rank: {summary['identifiability_effective_rank']} / {len(r['unknowns'])}",
            f"- Wall time [s]: {_fmt(summary['elapsed_s'])}",
            "",
            "Input values are stored in `input.json` and `input.csv`.",
            "Estimated parameters are stored in `parameters.csv`.",
            "Observed/predicted/residual time series are stored in `timeseries.csv`.",
            "Fit and identifiability diagnostics are stored in `diagnostics.json` and `sensitivity.csv`.",
        ]
        if ISAVE_FIELD.value:
            report.append("Final field and observation-time fields are stored in `field.npz`.")
        (run_dir / "report.md").write_text("\n".join(report), encoding="utf-8")

        refresh_inverse_runs()
        ISAVE_STATUS.value = f"<span style='color:green'><b>Saved:</b> {html.escape(str(run_dir))}</span>"
        return run_dir
    except Exception as e:
        ISAVE_STATUS.value = f"<span style='color:#b00020'><b>Save error:</b> {html.escape(str(e))}</span>"
        return None


# -----------------------------
# Load / refresh functions
# -----------------------------
def refresh_forward_runs(_=None):
    choices = _saved_run_choices(FRUN_DIR.value, "Forward")
    FRUN_SELECT.options = choices if choices else [("No saved forward runs", None)]


def refresh_inverse_runs(_=None):
    choices = _saved_run_choices(IRUN_DIR.value, "Inverse")
    IRUN_SELECT.options = choices if choices else [("No saved inverse runs", None)]


def load_forward_run(_=None):
    p = FRUN_SELECT.value
    if not p:
        FSAVE_STATUS.value = "<span style='color:#b36b00'>No saved forward run selected.</span>"
        return
    try:
        rec = json.loads((Path(p) / "input.json").read_text(encoding="utf-8"))
        if rec.get("mode") != "Forward":
            raise ValueError("Selected record is not a Forward run.")
        restore_forward_ui(rec["ui_state"])
        LAST_FORWARD_RESULT.update(result=None, config=None, ui_state=None, elapsed_s=None, run_started_at=None)
        FSAVE.disabled = True
        with FRESULT_SUMMARY:
            clear_output(wait=True)
            print("Saved input restored. Press 'Run forward' to reproduce this calculation.")
        with FDETAIL_SUMMARY:
            clear_output(wait=True)
        render_forward_input_summary(collect_forward())
        FRECORD_ACC.selected_index = 0
        FSAVE_STATUS.value = f"<span style='color:green'><b>Loaded input:</b> {html.escape(str(p))}</span>"
    except Exception as e:
        FSAVE_STATUS.value = f"<span style='color:#b00020'><b>Load error:</b> {html.escape(str(e))}</span>"


def load_inverse_run(_=None):
    p = IRUN_SELECT.value
    if not p:
        ISAVE_STATUS.value = "<span style='color:#b36b00'>No saved inverse run selected.</span>"
        return
    try:
        rec = json.loads((Path(p) / "input.json").read_text(encoding="utf-8"))
        if rec.get("mode") != "Inverse":
            raise ValueError("Selected record is not an Inverse run.")
        restore_inverse_ui(rec["ui_state"])
        LAST_INVERSE_RESULT.update(result=None, config=None, ui_state=None, elapsed_s=None, run_started_at=None, sensitivity=None)
        ISAVE.disabled = True
        with IRESULT_SUMMARY:
            clear_output(wait=True)
            print("Saved input restored. Press 'Run inverse' to reproduce this calculation.")
        with IDETAIL_SUMMARY:
            clear_output(wait=True)
        cfg = collect_inverse()
        render_inverse_input_summary(cfg, serialize_inverse_ui())
        IRECORD_ACC.selected_index = 0
        ISAVE_STATUS.value = f"<span style='color:green'><b>Loaded input:</b> {html.escape(str(p))}</span>"
    except Exception as e:
        ISAVE_STATUS.value = f"<span style='color:#b00020'><b>Load error:</b> {html.escape(str(e))}</span>"


# -----------------------------
# v2.3 summary + record widgets
# -----------------------------
FINPUT_SUMMARY = W.Output()
FRESULT_SUMMARY = W.Output()
FDETAIL_SUMMARY = W.Output()

FRECORD_ACC = W.Accordion(children=[FINPUT_SUMMARY, FRESULT_SUMMARY, FDETAIL_SUMMARY])
for i, title in enumerate(["Input summary", "Result summary", "Time-series output"]):
    FRECORD_ACC.set_title(i, title)
FRECORD_ACC.selected_index = 0

FRUN_DIR = W.Text(value=str(Path.cwd() / "runs"), description="Run folder", layout=W.Layout(width="650px"))
FRUN_LABEL = W.Text(value="", description="Label", placeholder="optional")
FSAVE_FIELD = W.Checkbox(value=True, description="save field.npz", indent=False)
FAUTOSAVE = W.Checkbox(value=False, description="auto-save after run", indent=False)
FSAVE = W.Button(description="Save last run", icon="save", button_style="info", disabled=True)
FREFRESH_RUNS = W.Button(description="Refresh saved runs", icon="refresh")
FRUN_SELECT = W.Dropdown(options=[("No saved forward runs", None)], description="Saved run", layout=W.Layout(width="650px"))
FLOAD = W.Button(description="Load input", icon="folder-open")
FSAVE_STATUS = W.HTML()

FSAVE_PANEL = W.VBox([
    W.HTML("<b>Save / load reproducibility record</b><br><span style='color:#666'>Save captures the input snapshot used for the completed run. Load restores input values; press Run again to reproduce.</span>"),
    FRUN_DIR,
    W.HBox([FRUN_LABEL, FSAVE_FIELD, FAUTOSAVE]),
    W.HBox([FSAVE, FREFRESH_RUNS]),
    FRUN_SELECT,
    W.HBox([FLOAD, FSAVE_STATUS]),
])

IINPUT_SUMMARY = W.Output()
IRESULT_SUMMARY = W.Output()
IDETAIL_SUMMARY = W.Output()

IRECORD_ACC = W.Accordion(children=[IINPUT_SUMMARY, IRESULT_SUMMARY, IDETAIL_SUMMARY])
for i, title in enumerate(["Input summary", "Result summary", "Observed / predicted / residual"]):
    IRECORD_ACC.set_title(i, title)
IRECORD_ACC.selected_index = 0

IRUN_DIR = W.Text(value=str(Path.cwd() / "runs"), description="Run folder", layout=W.Layout(width="650px"))
IRUN_LABEL = W.Text(value="", description="Label", placeholder="optional")
ISAVE_FIELD = W.Checkbox(value=True, description="save field.npz", indent=False)
IAUTOSAVE = W.Checkbox(value=False, description="auto-save after run", indent=False)
ISAVE = W.Button(description="Save last run", icon="save", button_style="info", disabled=True)
IREFRESH_RUNS = W.Button(description="Refresh saved runs", icon="refresh")
IRUN_SELECT = W.Dropdown(options=[("No saved inverse runs", None)], description="Saved run", layout=W.Layout(width="650px"))
ILOAD = W.Button(description="Load input", icon="folder-open")
ISAVE_STATUS = W.HTML()

ISAVE_PANEL = W.VBox([
    W.HTML("<b>Save / load reproducibility record</b><br><span style='color:#666'>Input, estimates, fit diagnostics, sensitivity diagnostics, and time series are stored together.</span>"),
    IRUN_DIR,
    W.HBox([IRUN_LABEL, ISAVE_FIELD, IAUTOSAVE]),
    W.HBox([ISAVE, IREFRESH_RUNS]),
    IRUN_SELECT,
    W.HBox([ILOAD, ISAVE_STATUS]),
])


# -----------------------------
# Replace old run callbacks
# -----------------------------
RUN.on_click(run_forward_clicked, remove=True)
IRUN.on_click(run_inverse_clicked, remove=True)


def run_forward_v23_clicked(_):
    FSTATUS.value = "<b>Running...</b>"
    with FOUT:
        clear_output(wait=True)
        try:
            cfg = collect_forward()
            ui_state = serialize_forward_ui()
            render_forward_input_summary(cfg)

            started = datetime.now().astimezone().isoformat(timespec="seconds")
            t0 = time.perf_counter()
            res = solve_forward(cfg)
            elapsed = time.perf_counter() - t0

            LAST_FORWARD_RESULT.update(
                result=res,
                config=_jsonable(cfg),
                ui_state=ui_state,
                elapsed_s=elapsed,
                run_started_at=started,
            )

            plot_forward(res)
            print(f"Wall time: {elapsed:.3f} s")
            render_forward_result_summary(res, elapsed)
            FRECORD_ACC.selected_index = 1
            FSAVE.disabled = False
            FSTATUS.value = "<span style='color:green'><b>Completed.</b></span>"

            if FAUTOSAVE.value:
                save_forward_run()
        except Exception as e:
            print(type(e).__name__ + ":", e)
            FSTATUS.value = f"<span style='color:#b00020'><b>Error:</b> {html.escape(str(e))}</span>"


def run_inverse_v23_clicked(_):
    ISTATUS.value = "<b>Running inverse...</b>"
    with IOUT:
        clear_output(wait=True)
        try:
            if OBS_SOURCE.value == "Synthetic" and not DATA_TEXT.value.strip():
                raise ValueError("Generate synthetic data first (or load the verified example).")

            cfg = collect_inverse()
            ui_state = serialize_inverse_ui()
            render_inverse_input_summary(cfg, ui_state)

            started = datetime.now().astimezone().isoformat(timespec="seconds")
            t0 = time.perf_counter()
            r = run_inverse(cfg)
            elapsed = time.perf_counter() - t0

            LAST_INVERSE_RESULT.update(
                result=r,
                config=_jsonable(cfg),
                ui_state=ui_state,
                elapsed_s=elapsed,
                run_started_at=started,
                sensitivity=None,
            )

            plot_inverse(r)
            if SENS_AUTO.value:
                analyze_last_fit()
            sens = render_inverse_result_summary(r, elapsed)
            LAST_INVERSE_RESULT["sensitivity"] = sens
            IRECORD_ACC.selected_index = 1

            print(f"Wall time: {elapsed:.3f} s")
            d = r["diagnostics"]
            ISTATUS.value = f"<span style='color:{d['color']}'><b>Optimization completed — fit {d['grade']}.</b></span>"
            ISAVE.disabled = False

            if IAUTOSAVE.value:
                save_inverse_run()
        except Exception as e:
            print(type(e).__name__ + ":", e)
            ISTATUS.value = f"<span style='color:#b00020'><b>Error:</b> {html.escape(str(e))}</span>"


RUN.on_click(run_forward_v23_clicked)
IRUN.on_click(run_inverse_v23_clicked)

FSAVE.on_click(save_forward_run)
FREFRESH_RUNS.on_click(refresh_forward_runs)
FLOAD.on_click(load_forward_run)

ISAVE.on_click(save_inverse_run)
IREFRESH_RUNS.on_click(refresh_inverse_runs)
ILOAD.on_click(load_inverse_run)

FRUN_DIR.observe(refresh_forward_runs, names="value")
IRUN_DIR.observe(refresh_inverse_runs, names="value")

refresh_forward_runs()
refresh_inverse_runs()

# Show current default inputs before any run.
try:
    render_forward_input_summary(collect_forward())
except Exception:
    pass

try:
    render_inverse_input_summary(collect_inverse(), serialize_inverse_ui())
except Exception:
    pass


# -----------------------------
# Rebuild panels with v2.3 record sections
# -----------------------------
FORWARD_PANEL = W.VBox([
    W.HTML("<h3>Forward simulation</h3>"),
    METHOD,
    DIM,
    FACC,
    W.HBox([RUN, FSTATUS]),
    FOUT,
    W.HTML("<h3>Input / result record</h3>"),
    FRECORD_ACC,
    FSAVE_PANEL,
    W.HTML("<h3>What should I ask AI?</h3>"),
    FQUEST,
])

INVERSE_PANEL = W.VBox([
    W.HTML("<h3>Inverse: 1D / 2D FVM</h3>"),
    IDIM,
    IACC,
    W.HBox([IRUN, ISTATUS]),
    IOUT,
    W.HTML("<h3>Input / result record</h3>"),
    IRECORD_ACC,
    ISAVE_PANEL,
    W.HTML("<h3>What should I ask AI?</h3>"),
    IQUEST,
])

print("v2.3 record layer loaded: input summaries + result summaries + Save Run + Load Run.")

In [ ]:
# Assemble and display the app
APP=W.Tab(children=[FORWARD_PANEL,INVERSE_PANEL])
APP.set_title(0,"Forward")
APP.set_title(1,"Inverse 1D/2D")
display(APP)


## v2.3 interpretation / limits

1. **Synthetic round-trip verification is now the default inverse workflow.** The built-in 1D example generates observations with the same FVM forward engine using $D_{\rm true}=10^{-10}\,\mathrm{m^2/s}$ and fixed Robin $h=2\times10^{-8}\,\mathrm{m/s}$, then perturbs the inverse initial guess. This distinguishes a numerical inverse failure from an experimental-model mismatch.
2. **Optimizer convergence and fit quality are reported separately.** `success=True` only means the least-squares stopping condition was satisfied; it is not proof of a good fit or a valid physical model.
3. **Bound proximity is explicitly flagged.** An estimated parameter driven to a lower/upper bound is treated as a diagnostic, not as a successful physical estimate.
4. **Diagonal anisotropy is axis-aligned.** If material axes are rotated relative to the specimen/grid, the next model is $\mathbf D=\mathbf Q\mathbf D_{\rm mat}\mathbf Q^T$.
5. **Anisotropic forward and inverse use FVM + Picard.** The verified isotropic FDM/FVM/FEM backends remain intact.
6. The inverse observation operator is the domain mean $\bar u(t)$, matching gravimetric mean-moisture experiments when $u=M$.
7. A locally full-rank optimizer Jacobian is **not** proof of global identifiability or uniqueness. Profile likelihood, multi-start fitting, sensitivity analysis, and additional experiments remain important.
8. Current Robin coefficients are constant. Time/state/correlation-dependent Robin laws remain a later boundary-law layer.
9. 3D inverse remains deliberately excluded because repeated 3D nonlinear forward solves are computationally expensive and usually require stronger identifiability evidence.
10. **Sensitivity / Identifiability panel:** the final optimizer Jacobian is converted back to prediction sensitivities and parameter-scaled for comparison. It reports time-resolved sensitivity curves, RMS sensitivity strength, sensitivity-column cosine similarity, normalized singular values, and an effective local rank. The resulting GOOD/REVIEW/POOR label is a heuristic for **local practical identifiability only**; it is not a proof of structural or global identifiability.
11. **Run records are reproducibility snapshots.** The input state is frozen at run start; changing widgets afterward does not alter the saved configuration of the completed run.
12. **Load Run restores inputs, not trust.** A loaded run must be executed again to reproduce the result under the current Python/package environment.
13. **Field output is compressed and optional.** CSV/JSON records remain lightweight; `field.npz` can be disabled for large 2D/3D runs.

### Public-release note

This notebook is part of the v2.3 research-software release. Citation metadata are in `CITATION.cff`; development-transparency information is in `AI_DISCLOSURE.md`. The numerical backends should be independently verified for any new scientific application.
